# Buoyancy budget profiles on height-above-bottom (hab) coordinates

Short version of `buoyancy_profiles_rrexnum`: profiles of the buoyancy budget averaged over regions selected by

* the sign of the bottom vertical velocity $w_{bot}$, and
* the sign of the mean stratification $\overline{N^2_{BBL}}$ in the bottom boundary layer (BBL).

The BBL is the near-bottom weakly stratified layer. Its thickness is diagnosed with the **curvature method** (Lorbacher et al. 2006), from the shape of the $N^2$ profile of each column:
$$h_{BBL} = hab\left[\max\left(\frac{\partial^2 N^2}{\partial z^2}\right)\right]$$
i.e. the height above bottom where the second vertical derivative of $N^2$ is maximum (the sharp transition from the weakly stratified layer to the stratified interior). $\overline{N^2_{BBL}}$ is the mean of $N^2$ between the bottom and $h_{BBL}$.

Instead of averaging sigma layers, each water column is first **interpolated linearly onto a common grid of height above bottom** (`hab = z - z_bottom`, 0 = sea floor), then the columns of each region are averaged level by level.
Budget terms are buoyancy tendencies [m s$^{-3}$].

**Stratification budget.** Since $N^2 = \partial b/\partial z$, the tendency of the stratification is the vertical derivative of the buoyancy budget:
$$\frac{\partial N^2}{\partial t} = \frac{\partial b_{adv}}{\partial z} + \frac{\partial b_{rhs}}{\partial z}$$
(advection and mixing/forcing terms), further split into mean-flow and eddy advection. The derivatives are taken on the native vertical grid and then remapped to hab, like every other field. They are in s$^{-3}$; positive values strengthen the stratification.

**Eddy advection, vertical vs horizontal.** The eddy part of the advection, $-\overline{\mathbf{u'}\cdot\nabla b'} = -\nabla\cdot\overline{\mathbf{u'}b'}$, is split into
* a vertical part $-\partial_z \overline{w'b'}$, with $\overline{w'b'} = \overline{wb} - \overline{w}\,\overline{b}$ from the eddy-diagnostics file (`_diags_eddy`) and the mean fields of the average file, and
* a horizontal part, the remainder (total eddy $-$ vertical), and, as a cross-check, the horizontal part computed **directly** from the fluxes of the eddy file, $-(\partial_x \overline{u'b'} + \partial_y \overline{v'b'})$ (dashed lines). The remainder contains the error of the mean-advection term (a difference of two large, nearly cancelling terms); the direct version does not. If the dashed and solid lines disagree, trust the direct one.

The same split is done for the stratification budget (one more $\partial_z$).

**What produces $\overline{w'b'}$? Baroclinic instability or something else?** A positive $\overline{w'b'}$ (eddy APE $\to$ EKE) also comes from convection and frontal circulations, so several independent diagnostics are computed per region:
* *energetics*: baroclinic conversion $\overline{w'b'}$ against the barotropic (horizontal Reynolds stress) and vertical-shear conversions of EKE;
* *eddy APE production* $-\overline{\mathbf{u'}b'}\cdot\nabla b$, split into its horizontal part (positive for down-gradient fluxes, the baroclinic signature) and vertical part $-\overline{w'b'}N^2$; their sum is zero for purely along-isopycnal (adiabatic) eddy fluxes;
* *along-isopycnal test*: ratio of $\overline{w'b'}$ to the value it would have if the flux were along mean isopycnals, $\overline{w'b'}_{adiab} = -\overline{\mathbf{u'_h}b'}\cdot\nabla_h b / N^2$ (ratio $\approx 1$: slumping along isopycnals);
* *mean-state criteria*: Eady growth rate $0.31\,M^2/N$, balanced Richardson number $N^2f^2/M^4$, ratio of bottom slope to isopycnal slope, mean-flow Rossby number, and BBL deformation radius over grid spacing ($M^2 = |\nabla_h b|$).

Not computed here (they need snapshots or a separate calculation): the scale/frequency decomposition of $w'b'$ and the linear stability analysis of the mean profile.

In [ ]:
# --- Setup -----------------------------------------------------------------------------------
%matplotlib inline
import sys, warnings, gc

# ROMS/CROCO python tools (path on the original Linux machine; see buoyancy_profiles_rrexnum.ipynb)
sys.path.append("/home/gula/Desktop/Work_capella/Python/Python_Modules_p3/")
from Modules import *          # np, plt, tools, toolsF, load, ...
from Modules_gula import *     # var (R_vars_gula)
from matplotlib.ticker import NullFormatter

# --- Configuration ---------------------------------------------------------------------------
mysimul = 'rrexnumsb200_RSUP5_NOFILT_avg_mean.00010-00229'    # time-mean run + its _wdia diagnostics
time_his = 0                                                       # time index in the average files

# Zoomed sub-domain in grid indices (j_sec +/- dy_sec, i_sec +/- dx_sec); sigma levels 1..300, 1 = bottom
i_sec, dx_sec = 500, 450
j_sec, dy_sec = 400, 350
coordinates = ' [{},{},{},{},[1,300,1]] '.format(max(j_sec - dy_sec, 0), j_sec + dy_sec,
                                                  max(i_sec - dx_sec, 0), i_sec + dx_sec)

# Regions
TOPO_MIN = 500.      # keep ocean columns deeper than this [m]
TOPO_MAX = None      # and shallower than this [m] (None: no upper limit)
HBBL_SEARCH_MAX = 300.   # the curvature maximum is searched for below this height above bottom [m]

# Height-above-bottom grid [m]
DH, HAB_MAX = 5., 300.
hab_grid = np.arange(0., HAB_MAX + DH, DH)

# Statistic and figures
STRAT_XLIM = None    # x range of the stratification-budget panels [s-3], e.g. (-5e-13, 5e-13); None = automatic
STAT = 'mean'        # 'mean' or 'median' over the columns of a region
HAB_PLOT = 200.      # upper limit of the height axis [m]
BMASK = 1e-8         # points with |b_adv| > BMASK [m s-3] are masked (NaN) in ALL budget terms (b_adv, b_rhs, b_adv_mean, b_eddy, ...), profiles, tables and maps (None: no mask)
fifig = './'

In [ ]:
# --- Load the zoomed time-mean simulation and its buoyancy-budget diagnostics (_wdia) ------------
simul0 = load(mysimul, time=time_his, output=False)
simul = load(simul=simul0.simul + coordinates + format(simul0.time), output=False)
simul_wdia = load(simul=simul.simul + '_wdia' + coordinates + format(time_his), output=False)

In [ ]:
F32 = np.float32
CH = 20000            # columns processed at a time (bounds the size of the temporary arrays)


class Remap:
    """Linear interpolation of columns from their own heights onto `hgrid` (height above bottom).

    hab   : (ncol, nk) height above bottom of the levels of each column (increasing along nk)
    hgrid : (nh,) target heights
    The bracketing levels (int16) and the weights (float32) are computed once, in chunks of columns,
    with ONE vectorised searchsorted per chunk (columns are separated by an offset so that the flattened
    heights are sorted); they are shared by all the fields.  The weight is NaN where the target height
    lies outside [hab[:, 0], hab[:, -1]] (no extrapolation), so that the result is NaN there.
    Calling remap(f, sl) interpolates the (n, nk) field `f` of the columns `sl`.
    """

    def __init__(self, hab, hgrid, chunk=CH):
        ncol, nk = hab.shape
        nh = len(hgrid)
        self.k = np.empty((ncol, nh), np.int16)
        self.w = np.empty((ncol, nh), np.float32)
        big = 2. * (float(np.nanmax(hab)) + float(np.max(hgrid)) + 1.)
        for s in range(0, ncol, chunk):
            h = hab[s:s + chunk].astype(np.float64)
            n = h.shape[0]
            off = (np.arange(n, dtype=np.float64) * big)[:, None]
            idx = np.searchsorted((h + off).ravel(), (hgrid[None, :] + off).ravel(), side='right') - 1
            k = np.clip(idx.reshape(n, nh) - (np.arange(n) * nk)[:, None], 0, nk - 2)    # level just below each target
            h0 = np.take_along_axis(h, k, axis=1)
            h1 = np.take_along_axis(h, k + 1, axis=1)
            w = (hgrid[None, :] - h0) / (h1 - h0)
            w[(hgrid[None, :] < h[:, :1]) | (hgrid[None, :] > h[:, -1:])] = np.nan
            self.k[s:s + n], self.w[s:s + n] = k, w

    def __call__(self, f, sl):
        k = self.k[sl].astype(np.intp)
        f0 = np.take_along_axis(f, k, axis=1)
        f1 = np.take_along_axis(f, k + 1, axis=1)
        return f0 + self.w[sl] * (f1 - f0)


def ld(name, sim, dtype=F32, **kw):
    """Variable as a contiguous array of type `dtype`, cut at the first `nku` levels (hab <= HAB_MAX + margin).

    Everything below 300 m above the bottom is kept, the rest of the water column is never needed here.
    Arrays with a single level (depths=[1]) are returned as they are.
    """
    d = var(name, sim, **kw).data
    if d.ndim == 3 and d.shape[2] > 1:
        nk = d.shape[2]
        d = d[:, :, :nku] if nk == nz_full else d[:, :, :nku + 1] if nk == nz_full + 1 else d
    return np.array(d, dtype=dtype)


def bbl_height_curvature(N2, hgrid, hab_max=None):
    """BBL thickness [m] by the curvature method (Lorbacher et al. 2006).

    h_BBL = hab at which d2(N^2)/dz2 is maximum, for each column.
    N2    : (ncol, nh) N^2 on the uniform height-above-bottom grid `hgrid`; NaN where undefined
    hab_max : only heights <= hab_max are searched (None: the whole profile)
    The second derivative is the centred second difference, so it exists only at heights where
    N^2 is known at the three neighbouring grid points. Returns NaN for columns with none.
    """
    dh = hgrid[1] - hgrid[0]
    d2 = (N2[:, 2:] - 2. * N2[:, 1:-1] + N2[:, :-2]) / dh**2        # curvature at hgrid[1:-1]
    if hab_max is not None:
        d2 = np.where(hgrid[None, 1:-1] <= hab_max, d2, np.nan)
    valid = ~np.all(np.isnan(d2), axis=1)
    k = np.argmax(np.where(np.isnan(d2), -np.inf, d2), axis=1)      # nan-safe argmax
    return np.where(valid, hgrid[1:-1][k], np.nan)


def get_brhs(simul, T, S, t_rhs, s_rhs, z_r, z_w):
    """Buoyancy tendency from temperature / salinity tendencies: g * (alpha dT/dt - beta dS/dt)."""
    if 'NONLIN_EOS' in simul.cpp:
        [alpha, beta] = toolsF.alfabeta_3d(T, S, z_r, z_w, simul.rho0)
    else:
        [alpha, beta] = np.array([simul.Tcoef, simul.Scoef]) / simul.rho0
    return simul.g * (alpha * t_rhs - beta * s_rhs)


def mean_advection():
    """-u.grad(b) of the mean flow from the temperature / salinity advection terms, (nx, ny, nku) float32."""
    [z_r, z_w] = tools.get_depths(simul)
    t = var('temp', simul, depths=[0])
    [TX, TY, TV] = t.get_tracer_advection(simul)             # last axis: 0 = temperature, 1 = salinity
    advT = np.array((TX[..., 0] + TY[..., 0] + TV[..., 0])[:, :, :nku])
    advS = np.array((TX[..., 1] + TY[..., 1] + TV[..., 1])[:, :, :nku])
    del TX, TY, TV, t                                        # the largest arrays of the notebook: free them at once
    if 'NONLIN_EOS' in simul.cpp:
        # alpha, beta must be computed on the FULL water column (the depth used for the pressure is measured from the surface)
        [alpha, beta] = toolsF.alfabeta_3d(var('temp', simul).data, var('salt', simul).data, z_r, z_w, simul.rho0)
        alpha, beta = alpha[:, :, :nku], beta[:, :, :nku]
    else:
        [alpha, beta] = np.array([simul.Tcoef, simul.Scoef]) / simul.rho0
    return (simul.g * (alpha * advT - beta * advS)).astype(F32)


BAD = None     # model levels (nx, ny, nku): True where |b_adv| > BMASK (set when b_adv is loaded)
HM = None      # hab levels (nx, ny, nhab): True where the remapped b_adv is NaN (masked)


def mask_b(a):
    """NaN (in place) where |b_adv| > BMASK, on model levels. The mask is defined by b_adv ONLY and applied to every term."""
    if BAD is not None:
        a[BAD] = np.nan
    return a


def mask_h(a):
    """Same mask on hab levels (where the remapped b_adv is masked)."""
    if HM is not None:
        a[HM] = np.nan
    return a


In [ ]:
# --- Water columns used: ocean points deeper than TOPO_MIN (and shallower than TOPO_MAX, if set) ------------------------------------
nx, ny = simul.topo.shape
keep = (simul.mask.reshape(-1) > 0) & (simul.topo.reshape(-1) > TOPO_MIN)
if TOPO_MAX is not None:
    keep &= simul.topo.reshape(-1) < TOPO_MAX
cols = np.flatnonzero(keep)
ncol = len(cols)

wbot = var('w', simul, depths=[1]).data.reshape(-1)[cols]    # w at the lowest level (1 = bottom)

# Height above bottom of the levels of the selected columns.
# Only the lowest levels are kept: those with hab <= HAB_MAX (+ 2 for the bracketing of the top of the grid), `nku` levels
# in the thickest column. All 3D fields are cut at `nku` levels when they are loaded (ld) -> 3-4 times less memory.
# The end w-levels (sea floor, surface) are left out, so w-level fields (N^2, AKt) are only
# interpolated between the first and last retained interior w-level.
[z_r, z_w] = tools.get_depths(simul)
nz_full = z_r.shape[2]
hab_r = (z_r - z_w[:, :, :1]).reshape(nx * ny, nz_full)[cols]
nku = int(min(nz_full, (hab_r <= HAB_MAX).sum(axis=1).max() + 2))
hab_r = hab_r[:, :nku].astype(F32)                                                             # rho-levels        (ncol, nku)
hab_w = (z_w - z_w[:, :, :1]).reshape(nx * ny, nz_full + 1)[cols][:, 1:nku].astype(F32)        # interior w-levels (ncol, nku-1)
del z_r, z_w
nz = nku                                                                                       # number of retained rho-levels
dz_r = np.diff(hab_r, axis=1)                                                                  # spacing between rho-levels (ncol, nku-1)
print('%d of %d sigma levels kept (hab <= %g m)' % (nku, nz_full, HAB_MAX))

remap_r = Remap(hab_r, hab_grid)
remap_w = Remap(hab_w, hab_grid)
remap_mid = Remap(0.5 * (hab_w[:, 1:] + hab_w[:, :-1]), hab_grid)          # d2/dz2: between interior w-levels


def _chunks():
    for s in range(0, ncol, CH):
        yield slice(s, min(s + CH, ncol))


def to_hab(field):
    """(nx, ny, nk) field on rho-levels (nk = nku) or w-levels (nk = nku+1) -> (ncol, nh) float32 on hab_grid."""
    nk = field.shape[-1]
    assert nk in (nz, nz + 1), 'unexpected number of vertical levels: %d' % nk
    fl = field.reshape(nx * ny, nk)
    out = np.empty((ncol, len(hab_grid)), F32)
    for sl in _chunks():
        f = fl[cols[sl]].astype(F32, copy=False)
        out[sl] = remap_r(f, sl) if nk == nz else remap_w(f[:, 1:], sl)
    return out


def ddz_to_hab(field):
    """Vertical derivative d/dz of a rho-level (nx, ny, nku) field -> (ncol, nh) float32 on hab_grid.

    Centred difference between neighbouring rho-levels on the native grid (true layer spacing),
    which places the result on the interior w-levels, like N^2; then remapped to hab.
    """
    assert field.shape[-1] == nz, 'expects a field on rho-levels'
    fl = field.reshape(nx * ny, nz)
    out = np.empty((ncol, len(hab_grid)), F32)
    for sl in _chunks():
        f = fl[cols[sl]].astype(F32, copy=False)
        out[sl] = remap_w(np.diff(f, axis=1) / dz_r[sl], sl)
    return out


def ddz2_to_hab(field):
    """Second vertical derivative d2/dz2 of a rho-level (nx, ny, nku) field -> (ncol, nh) float32 on hab_grid.

    Two successive differences on the native grid: the first lands on the interior w-levels,
    the second between them (hence the extra remap with `remap_mid`).
    """
    assert field.shape[-1] == nz, 'expects a field on rho-levels'
    fl = field.reshape(nx * ny, nz)
    out = np.empty((ncol, len(hab_grid)), F32)
    for sl in _chunks():
        f = fl[cols[sl]].astype(F32, copy=False)
        d1 = np.diff(f, axis=1) / dz_r[sl]
        out[sl] = remap_mid(np.diff(d1, axis=1) / np.diff(hab_w[sl], axis=1), sl)
    return out


print(ncol, 'columns selected')


In [ ]:
# --- Regions: sign of w_bot x sign of the BBL stratification -------------------------------------
H = {}                                                       # fields on hab_grid: name -> (ncol, nh)
H['N2'] = to_hab(ld('bvf', simul))

# BBL thickness of every column (curvature method) and the mean N^2 within it
hbbl = bbl_height_curvature(H['N2'], hab_grid, hab_max=HBBL_SEARCH_MAX)
in_bbl = (hab_grid[None, :] > 0) & (hab_grid[None, :] <= hbbl[:, None])     # all False where hbbl is NaN
with warnings.catch_warnings():
    warnings.simplefilter('ignore', RuntimeWarning)          # columns without BBL: all-NaN slices
    N2_bbl = np.nanmean(np.where(in_bbl, H['N2'], np.nan), axis=1)

# Columns where the curvature is undefined (N^2 known at < 3 heights) get N2_bbl = NaN and
# belong to none of the four sign classes below (but stay in 'all').
print('h_BBL: median %.0f m, 10-90%% range %.0f-%.0f m; %d columns without BBL'
      % (*np.nanpercentile(hbbl, [50, 10, 90]), np.isnan(hbbl).sum()))

regions = {
    'all (topo>%d m)' % TOPO_MIN: np.ones(ncol, dtype=bool),
    'w_bot<0, N2_bbl>0': (wbot < 0) & (N2_bbl > 0),
    'w_bot>0, N2_bbl>0': (wbot > 0) & (N2_bbl > 0),
    'w_bot<0, N2_bbl<0': (wbot < 0) & (N2_bbl < 0),
    'w_bot>0, N2_bbl<0': (wbot > 0) & (N2_bbl < 0),
}
for name, sel in regions.items():
    print('{:>7d} columns : {}'.format(sel.sum(), name))

In [ ]:
# --- Remap the budget terms and diagnostics onto hab_grid, then average over each region ---------
# Buoyancy-budget terms on rho-levels (nx, ny, nku), one at a time: remapped (H) and differentiated in z (G), then freed.
bot = {}                                                           # lowest-level (sigma 1) values, for the maps
G = {}
for key, getter in [('b_adv', lambda: ld('b_adv', simul_wdia)),    # advection tendency (online); loaded first: it defines the mask
                    ('b_rhs', lambda: ld('b_rhs', simul_wdia)),    # -div(B): mixing term
                    ('b_adv_mean', mean_advection),                # advection by the mean flow
                    ('b_rate', lambda: ld('b_rate', simul_wdia))]: # online buoyancy tendency
    field = getter()
    if key == 'b_adv' and BMASK is not None:
        BAD = np.abs(field) > BMASK                                # mask defined by b_adv only ...
    field = mask_b(field)                                          # ... applied to all terms before remapping and differentiating
    if key != 'b_rate':
        H[key] = to_hab(field)
    G['d_' + key] = ddz_to_hab(field)                              # stratification budget: d(term)/dz  [s-3]
    bot[key] = np.array(field[:, :, 0])
    del field
    gc.collect()
del BAD
BAD = None
HM = np.isnan(H['b_adv']) if BMASK is not None else None         # same mask on hab levels
H['b_eddy'] = H['b_adv'] - H['b_adv_mean']                      # eddy part: -<u'.grad b'>  (NaN where b_adv is masked)
H['K_eff'] = to_hab(ld('diffusivity', simul_wdia))              # effective (online) diffusivity
H['K_KPP'] = to_hab(ld('AKt', simul))                           # KPP diffusivity

# Stratification budget: vertical derivative of each buoyancy term, dN2/dt = d(b_adv + b_rhs)/dz  [s-3]
G['d_b_eddy'] = G['d_b_adv'] - G['d_b_adv_mean']
for k in list(G):                                                # derivatives: same points as b_adv (no NaN spreading differences)
    mask_h(G[k])
G['d_b_sum'] = G['d_b_adv'] + G['d_b_rhs']                      # advection + mixing; compare with d_b_rate


In [ ]:
# --- Check of the b_adv mask: masked area [%] and pdf of b_adv ---------------------------------------------------------------------
# Points with |b_adv| > BMASK are masked (NaN) in all the budget terms, at EVERY model level (not only at the bottom).  Here the UNMASKED
# b_adv is reloaded to see how much of the domain this removes: at the lowest level, in the whole layer hab <= HAB_PLOT, and in the
# number of columns having at least one masked level in that layer.
if BMASK is None:
    print('BMASK = None: nothing is masked')
else:
    b3 = ld('b_adv', simul_wdia)                                                           # unmasked b_adv (nx, ny, nku)
    badv_bot_grid = b3[:, :, 0].astype(np.float64)                                         # lowest level, whole grid
    b3c = b3.reshape(nx * ny, nku)[cols]                                                   # selected columns (ncol, nku)
    del b3
    badv_bot = b3c[:, 0].astype(np.float64)
    layer = hab_r <= HAB_PLOT                                                              # levels of the layer shown in the figures (ncol, nku)
    fin3 = np.isfinite(b3c) & layer
    over3 = fin3 & (np.abs(b3c) > BMASK)
    area_c = (1. / (simul.pm * simul.pn)).reshape(-1)[cols].astype(np.float64)             # grid-cell area of the columns [m2]
    area_g = 1. / (simul.pm * simul.pn)
    fin = np.isfinite(badv_bot); over = fin & (np.abs(badv_bot) > BMASK)

    def pct(sel):
        """Masked [%] (weighted by grid-cell area) of the columns `sel`: (lowest level, whole layer, columns with >= 1 masked level)."""
        sel = sel & fin
        if sel.sum() == 0:
            return np.nan, np.nan, np.nan
        a = area_c[sel]
        bot_ = 100. * (a * over[sel]).sum() / a.sum()
        lay_ = 100. * (a[:, None] * over3[sel]).sum() / (a[:, None] * fin3[sel]).sum()
        any_ = 100. * (a * over3[sel].any(axis=1)).sum() / a.sum()
        return bot_, lay_, any_

    def counts(sel):
        """Number of points above the threshold in the columns `sel`: (lowest level, whole layer, columns with >= 1 masked level)."""
        sel = sel & fin
        return int(over[sel].sum()), int(over3[sel].sum()), int(over3[sel].any(axis=1).sum())

    rn = list(regions)                                                                     # same order as in the summary figure: All grid, B, A, C, D
    LAB = {rn[0]: 'All grid', rn[2]: 'Subdomain A', rn[1]: 'Subdomain B', rn[3]: 'Subdomain C', rn[4]: 'Subdomain D'}
    SC = {rn[0]: 'gray', rn[2]: 'orange', rn[1]: 'purple', rn[3]: 'red', rn[4]: 'blue'}       # colours of the subdomains (as in the summary figure)
    groups = [(n, regions[n]) for n in [rn[0], rn[2], rn[1], rn[3], rn[4]]]
    print('|b_adv| > BMASK = %g m s-3 is masked in all terms.  Number of points above the threshold (and masked area [%%], weighted by grid-cell area):' % BMASK)
    print('%-22s %8s | %-22s | %-26s | %-24s' % ('', 'columns', 'lowest level', 'layer hab<=%g m (levels)' % HAB_PLOT, 'columns with >=1 level'))
    for name, sel in groups:
        b_, l_, a_ = pct(sel); nb_, nl_, na_ = counts(sel)
        print('%-22s %8d | %8d (%7.3f %%) | %10d (%7.3f %%)    | %8d (%7.3f %%)' % (LAB[name], (sel & fin).sum(), nb_, b_, nl_, l_, na_, a_))
    print('(total number of levels in the layer, all columns: %d)' % fin3.sum())
    ocean2d = (simul.mask > 0) & np.isfinite(badv_bot_grid)                                # all ocean points of the zoom, whatever the depth
    print('%-26s %9d %14.3f   (lowest level, all ocean points of the zoomed grid, also shallower than TOPO_MIN)' % ('Whole zoomed grid', ocean2d.sum(),
          100. * (area_g * ocean2d * (np.abs(badv_bot_grid) > BMASK)).sum() / (area_g * ocean2d).sum()))
    q = np.nanpercentile(badv_bot, [0.1, 1, 50, 99, 99.9])
    print('b_adv at the lowest level [m s-3]: percentiles 0.1/1/50/99/99.9 = ' + ' / '.join('%.2e' % v for v in q) + ',  max |b_adv| = %.2e' % np.nanmax(np.abs(badv_bot)))
    print('b_adv in the layer hab <= %g m: max |b_adv| = %.2e' % (HAB_PLOT, np.nanmax(np.abs(np.where(fin3, b3c, np.nan)))))

    # masked fraction as a function of height above bottom (hab levels: includes the points NaN after the vertical interpolation)
    frac_hab = {n: 100. * np.mean(HM[sel], axis=0) for n, sel in groups if sel.sum() > 0}

    # the NTOP columns with the largest |b_adv| in the layer: table
    NTOP = 10
    amax = np.nanmax(np.where(fin3, np.abs(b3c), np.nan), axis=1)                          # largest |b_adv| of each column over the layer
    top = np.argsort(-np.where(np.isfinite(amax), amax, -1.))[:NTOP]
    topo_c = simul.topo.reshape(-1)[cols]
    letter_of = {rn[2]: 'A', rn[1]: 'B', rn[3]: 'C', rn[4]: 'D'}
    print('\nThe %d columns with the largest |b_adv| in the layer hab <= %g m:' % (NTOP, HAB_PLOT))
    print('%4s %6s %6s %8s %12s %9s %7s %8s' % ('rank', 'i', 'j', 'depth[m]', 'max|b_adv|', 'hab[m]', 'subdom.', 'n>BMASK'))
    for r, c in enumerate(top):
        kmax = int(np.nanargmax(np.where(fin3[c], np.abs(b3c[c]), np.nan)))
        sd = ''.join(letter_of[n] for n in letter_of if regions[n][c]) or '-'
        print('%4d %6d %6d %8.0f %12.2e %9.1f %7s %8d' % (r + 1, cols[c] // ny, cols[c] % ny, topo_c[c], amax[c], hab_r[c, kmax], sd, over3[c].sum()))

    with plt.rc_context({'font.size': 14}):
        fig, axs = plt.subplots(1, 2, figsize=(17, 6))
        # a) pdf of b_adv: density = counts / (N * bin width), bins of constant width in a symmetric-log sense
        xmax = max(float(np.nanmax(np.abs(badv_bot))), float(np.nanmax(np.abs(np.where(fin3, b3c, np.nan)))), 10. * BMASK)
        lin = 1e-13                                                                       # linear zone of the symlog axis [m s-3]
        pos = np.logspace(np.log10(lin), np.log10(xmax), 140)
        edges = np.unique(np.concatenate([-pos[::-1], [0.], pos]))
        centre = 0.5 * (edges[1:] + edges[:-1])
        for name, sel in groups:
            s_ = sel & fin
            if s_.sum() == 0:
                continue
            cnt, _ = np.histogram(badv_bot[s_], bins=edges)                               # solid: lowest level
            axs[0].plot(centre, np.where(cnt > 0, cnt / (s_.sum() * np.diff(edges)), np.nan), color=SC[name], lw=3 if name == rn[0] else 1.8, label=LAB[name])
        cnt, _ = np.histogram(b3c[fin3], bins=edges)                                      # dotted: all levels of the layer, all columns
        axs[0].plot(centre, np.where(cnt > 0, cnt / (fin3.sum() * np.diff(edges)), np.nan), color='k', ls=':', lw=2.5, label='all levels, hab <= %g m (all grid)' % HAB_PLOT)
        for sgn in (-1, 1):
            axs[0].axvline(sgn * BMASK, color='k', ls='--', lw=2)
        axs[0].axvspan(BMASK, 1e3 * xmax, color='k', alpha=0.08); axs[0].axvspan(-1e3 * xmax, -BMASK, color='k', alpha=0.08)
        b_, l_, a_ = pct(groups[0][1])
        axs[0].text(0.03, 0.04, r'$|b_{adv}|>$BMASK (area): %.2f %% at the lowest level, %.2f %% of the layer,' '\n'
                    r'%.2f %% of the columns have at least one masked level' % (b_, l_, a_), transform=axs[0].transAxes,
                    fontsize=12, bbox=dict(fc='w', ec='gray', alpha=0.9))
        axs[0].set_xscale('symlog', linthresh=lin); axs[0].set_yscale('log')
        axs[0].set_xlim(-xmax * 1.5, xmax * 1.5)
        axs[0].set_xlabel(r'$b_{adv}$ [m s$^{-3}$]'); axs[0].set_ylabel(r'pdf [(m s$^{-3}$)$^{-1}$]')
        axs[0].set_title('a) pdf of $b_{adv}$ (dashed: $\\pm$BMASK = %g, masked zone shaded)' % BMASK, fontsize=14)
        axs[0].grid(True, alpha=0.4); axs[0].legend(fontsize=11, loc='upper left')
        # b) masked fraction against height above bottom
        for name, f_ in frac_hab.items():
            axs[1].plot(f_, hab_grid, color=SC[name], lw=3 if name == rn[0] else 1.8, label=LAB[name])
        axs[1].set_ylim(0, HAB_PLOT); axs[1].set_xlabel('masked points [%]'); axs[1].set_ylabel('hab [m]')
        axs[1].set_title('b) masked fraction vs height above bottom', fontsize=14); axs[1].grid(True, alpha=0.4)
        plt.tight_layout()
        plt.savefig(fifig + 'bmask_check_' + simul.simul + '.png', dpi=200, bbox_inches='tight')

        # c) example profiles: the NTOP columns with the largest |b_adv| (unmasked b_adv on the model levels, hab of the model levels)
        figp, axp = plt.subplots(2, 5, figsize=(22, 9), sharey=True)
        for r, (c, ax) in enumerate(zip(top, axp.ravel())):
            h_ = hab_r[c]; b_ = b3c[c].astype(np.float64); inl = h_ <= HAB_PLOT
            ax.plot(b_[inl], h_[inl], 'k-o', ms=3, lw=1.5)
            bad_ = inl & (np.abs(b_) > BMASK)
            ax.plot(b_[bad_], h_[bad_], 'o', color='r', ms=6, label='masked (|b_adv| > BMASK)')
            for sgn in (-1, 1):
                ax.axvline(sgn * BMASK, color='r', ls='--', lw=1.5)
            ax.axvline(0, color='gray', lw=1)
            lim = 1.15 * max(amax[c], BMASK)
            ax.set_xlim(-lim, lim); ax.set_ylim(0, HAB_PLOT); ax.grid(True, alpha=0.4)
            ax.set_title('%d) i=%d, j=%d, depth %.0f m\nmax $|b_{adv}|$ = %.1e, %d levels masked' % (r + 1, cols[c] // ny, cols[c] % ny, topo_c[c], amax[c], over3[c].sum()), fontsize=11)
            ax.set_xlabel(r'$b_{adv}$ [m s$^{-3}$]', fontsize=12)
            ax.ticklabel_format(axis='x', style='sci', scilimits=(0, 0))
            if r % 5 == 0:
                ax.set_ylabel('hab [m]', fontsize=12)
            if r == 0:
                ax.legend(fontsize=9, loc='lower right')
        figp.suptitle('Unmasked $b_{adv}$ in the %d columns with the largest values (dashed: $\\pm$BMASK = %g)' % (NTOP, BMASK), fontsize=14)
        figp.tight_layout()
        figp.savefig(fifig + 'bmask_check_profiles_' + simul.simul + '.png', dpi=200, bbox_inches='tight')
    del badv_bot, badv_bot_grid, b3c, over3, fin3, layer, area_c, area_g, amax


In [ ]:
# --- Eddy advection split into vertical and horizontal parts ---------------------------------------
# Vertical eddy flux  w'b' = <wb> - <w><b>  (<wb> from the eddy-diagnostics file, rho-levels).
# NB: <b> must be defined exactly as the b used for <wb> in the model diagnostics.
# b_mean is kept in double precision (its vertical differences are tiny in the weakly stratified BBL); other 3D fields in float32.
simul_eddy = load(simul=simul.simul + '_diags_eddy' + coordinates + format(time_his), output=False)

wb_mean = ld('wb', simul_eddy, dtype=np.float64)
w_mean = ld('w', simul)
# b = -g (rho - rho0) / rho0 from the *mean* rho of the avg file (var('buoy') would recompute it from the mean T/S)
b_mean = -simul.g * (ld('rho', simul, dtype=np.float64) + 1000. - simul.rho0) / simul.rho0
wpbp = (wb_mean - w_mean * b_mean).astype(F32)                   # (nx, ny, nku)

# Sanity check: a b_mean with a different constant offset c leaves a spurious c * <w> in w'b'.
sub = (slice(None, None, 4), slice(None, None, 4))
ratio = np.nanmedian(np.abs(wpbp[sub])) / np.nanmedian(np.abs(wb_mean[sub]))
print('median |w\'b\'| / median |<wb>| = %.2g' % ratio)
if ratio > 0.5:
    print('WARNING: w\'b\' is as large as <wb> itself - check that <b> has the same definition '
          '(offset) as b in the eddy file')
del wb_mean              # w_mean and b_mean are reused for the instability diagnostics below

# Buoyancy budget: eddy advection = -div(u'b') ; vertical part -d(w'b')/dz, horizontal = rest
H['wpbp'] = to_hab(wpbp)                                         # the flux itself [m2 s-3]
H['w'] = to_hab(w_mean) * 86400.                                # mean vertical velocity [m/day] (summary figure)
H['b_eddy_vert'] = mask_h(-ddz_to_hab(wpbp))
H['b_eddy_hor'] = H['b_eddy'] - H['b_eddy_vert']

# Stratification budget: one more vertical derivative
G['d_b_eddy_vert'] = -ddz2_to_hab(wpbp)
G['d_b_eddy_vert'][np.isnan(H['b_eddy_vert'])] = np.nan          # same points as the buoyancy terms
G['d_b_eddy_hor'] = G['d_b_eddy'] - G['d_b_eddy_vert']
G['d_b_eddy_hor'][np.isnan(H['b_eddy_hor'])] = np.nan


In [ ]:
# --- Instability diagnostics: eddy fluxes, mean gradients, energy conversions ----------------------
# Every field is remapped to hab (H) as soon as it is computed; the 3D arrays are freed at the end of the cell.
[z_r, z_w] = tools.get_depths(simul)
z_r = z_r[:, :, :nku]
DZ = np.gradient(z_r, axis=2).astype(F32)                      # layer thickness (centred)
ZX = tools.u2rho(tools.diffx(z_r, simul.pm)).astype(F32)       # dz/dx along sigma
ZY = tools.v2rho(tools.diffy(z_r, simul.pn)).astype(F32)       # dz/dy along sigma
del z_r, z_w


def grad3d(f, comps='xyz'):
    """Gradient components of a rho-level (nx, ny, nku) field at constant z, at rho-points, float32.

    f_z : centred difference along the column with the true layer thickness;
    f_x, f_y : derivatives along the sigma surfaces, corrected for their slope: f_x - f_z * dz/dx.
    comps selects the components to compute ('x', 'y', 'z' in any combination); returns a list.
    """
    fz = np.gradient(f, axis=2) / DZ
    out = []
    for c in comps:
        if c == 'x':
            out.append((tools.u2rho(tools.diffx(f, simul.pm)) - fz * ZX).astype(F32))
        elif c == 'y':
            out.append((tools.v2rho(tools.diffy(f, simul.pn)) - fz * ZY).astype(F32))
        else:
            out.append(fz.astype(F32))
    return out


def eddy_cov(name, a_mean, b_mean_):
    """Eddy covariance <a'b'> = <ab> - <a><b>; <ab> from the eddy file. None if the file lacks it."""
    try:
        ab = ld(name, simul_eddy, dtype=np.float64)
    except Exception:
        print('no %s in the eddy file - the terms that need it are skipped' % name)
        return None
    return (ab - np.multiply(a_mean, b_mean_, dtype=np.float64)).astype(F32)


# Mean fields (rho-points) and their gradients
u_mean = tools.u2rho(ld('u', simul)).astype(F32)
v_mean = tools.v2rho(ld('v', simul)).astype(F32)
bx, by, bz = grad3d(b_mean)                                  # bz = N^2 of the mean rho
M2 = np.sqrt(bx**2 + by**2)                                  # |grad_h b|
H['M2'] = to_hab(M2)                                         # |grad_h b| on hab (Ri, regime diagram)
ux, uy, uz = grad3d(u_mean)
vx, vy, vz = grad3d(v_mean)

# Horizontal eddy buoyancy fluxes u'b', v'b'
upbp = eddy_cov('ub', u_mean, b_mean)
vpbp = eddy_cov('vb', v_mean, b_mean)
assert upbp is not None and vpbp is not None, 'ub and vb are needed in the eddy file'

# Horizontal eddy advection computed DIRECTLY from the fluxes: -(d_x u'b' + d_y v'b') at constant z.
# Alternative to the residual 'total eddy - vertical' (which carries the error of the mean-advection term).
ex = grad3d(upbp, 'x')[0]
ey = grad3d(vpbp, 'y')[0]
H['b_eddy_hor_direct'] = mask_h(to_hab(-(ex + ey)))
H['b_eddy_direct'] = H['b_eddy_vert'] + H['b_eddy_hor_direct']              # total eddy advection from the fluxes only
G['d_b_eddy_hor_direct'] = ddz_to_hab(-(ex + ey))
G['d_b_eddy_direct'] = G['d_b_eddy_vert'] + G['d_b_eddy_hor_direct']
G['d_b_eddy_hor_direct'][np.isnan(H['b_eddy_hor_direct'])] = np.nan
G['d_b_eddy_direct'][np.isnan(H['b_eddy_direct'])] = np.nan
del ex, ey

# Eddy APE production  -u'b'.grad(b)  [m2 s-5], split in horizontal and vertical parts
P_h = -(upbp * bx + vpbp * by)       # horizontal flux down the mean gradient: > 0  (baroclinic signature)
P_v = -wpbp * bz                     # vertical part: < 0 when w'b' > 0 (APE -> EKE)
H['P_h'], H['P_v'], H['P_tot'] = to_hab(P_h), to_hab(P_v), to_hab(P_h + P_v)   # P_tot = 0 if adiabatic
# Horizontal eddy buoyancy flux down the mean horizontal gradient  F_h = P_h / |grad_h b|  [m2 s-3]
with np.errstate(divide='ignore', invalid='ignore'):
    H['F_h'] = to_hab(P_h / np.where(M2 > 0, M2, np.nan))

    # Along-isopycnal test: w'b' that the horizontal fluxes would imply if u'b' . grad(b) = 0 (needs N^2 > 0)
    wpbp_adiab = np.where(bz > 0, P_h / bz, np.nan)
H['wpbp_adiab'] = to_hab(wpbp_adiab)
H['wpbp_cmp'] = to_hab(np.where(np.isnan(wpbp_adiab), np.nan, wpbp))      # w'b' where the test is defined
del P_h, P_v, wpbp_adiab, upbp, vpbp
gc.collect()

# Eddy kinetic energy sources (rho-points) [m2 s-3]; <ab> products from the eddy file
upup = eddy_cov('uu', u_mean, u_mean)
vpvp = eddy_cov('vv', v_mean, v_mean)
upvp = eddy_cov('uv', u_mean, v_mean)
if all(x is not None for x in (upup, vpvp, upvp)):
    H['C_hrs'] = to_hab(-upvp * (vx + uy) - upup * ux - vpvp * vy)       # barotropic: horizontal Reynolds stress
del upup, vpvp, upvp
upwp = eddy_cov('uw', u_mean, w_mean)
vpwp = eddy_cov('vw', v_mean, w_mean)
if all(x is not None for x in (upwp, vpwp)):
    H['C_vrs'] = to_hab(-upwp * uz - vpwp * vz)                          # vertical Reynolds stress (shear production)
del upwp, vpwp

# Mean-state criteria on hab. N^2 is the model's bvf (adiabatic gradient, H['N2']), M^2 = |grad_h b| of the mean b.
f_c = simul.f.reshape(-1)[cols].astype(F32)[:, None]
sx = np.gradient(simul.topo, axis=0) * simul.pm
sy = np.gradient(simul.topo, axis=1) * simul.pn
tan_th_cols = np.hypot(sx, sy).reshape(-1)[cols]             # bottom slope of the selected columns (regime diagram)
with np.errstate(divide='ignore', invalid='ignore'):
    N2h, M2h = H['N2'], H['M2']
    H['Ri'] = N2h * f_c**2 / M2h**2                                                   # balanced Richardson number (< 0 if N^2 < 0)
    stable = N2h > 0
    H['sigE'] = np.where(stable, 0.31 * M2h / np.sqrt(np.where(stable, N2h, 1.)), np.nan) * 86400.   # Eady growth [day-1]
    s_iso = np.where(stable, M2h / np.where(stable, N2h, 1.), np.nan)                 # isopycnal slope
    H['sbur'] = tan_th_cols[:, None] / s_iso                                          # bottom slope / isopycnal slope
del N2h, M2h, stable, s_iso
f3 = simul.f.astype(F32)[:, :, None]
H['Ro'] = to_hab((vx - uy) / f3)                                                      # mean-flow Rossby number

# Topostrophy of the bottom flow  tau = sign(f) k.(u x grad H) = sign(f) (u dH/dy - v dH/dx)  [m s-1], H = depth
# (> 0: shallower water on the right of the flow in the northern hemisphere); u, v = mean flow at the lowest level.
tau = np.sign(simul.f) * (u_mean[:, :, 0] * tools.v2rho(tools.diffy(simul.topo, simul.pn))
                          - v_mean[:, :, 0] * tools.u2rho(tools.diffx(simul.topo, simul.pm)))
tau_cols = tau.reshape(-1)[cols]

# Free the 3D arrays: only the (ncol, nh) fields in H and G are used below
del DZ, ZX, ZY, u_mean, v_mean, w_mean, b_mean, bx, by, bz, M2, ux, uy, uz, vx, vy, vz, wpbp, f3
gc.collect()


In [ ]:
# --- Statistic over the columns of each region, for every profile --------------------------------
def stat(a, axis=None):
    """nanmean (accumulated in double precision, the fields are float32) or nanmedian over `axis`."""
    return np.nanmean(a, axis=axis, dtype=np.float64) if STAT == 'mean' else np.nanmedian(a, axis=axis)
MEDIAN_KEYS = ['Ri', 'sigE', 'sbur', 'Ro']                      # ratio-type fields: always the median (means are dominated by outliers)
with warnings.catch_warnings():
    warnings.simplefilter('ignore', RuntimeWarning)             # empty regions -> NaN
    prof = {key: np.array([(np.nanmedian if key in MEDIAN_KEYS else stat)(field[sel], axis=0)
                           for sel in regions.values()])        # (n_regions, nh)
            for key, field in {**H, **G}.items()}

# Along-isopycnal ratio: w'b' / adiabatic prediction, from the region statistics (1 = purely along isopycnals)
with np.errstate(divide='ignore', invalid='ignore'):
    prof['R_adiab'] = prof['wpbp_cmp'] / prof['wpbp_adiab']

In [ ]:
# --- Stratification budget averaged over the BBL (0 < hab <= h_BBL) -------------------------------
# Per column: mean of each term over the BBL; then the statistic over the columns of each region.
terms = [('d_b_rhs', 'mixing'), ('d_b_adv', 'advection'), ('d_b_adv_mean', ' - mean'),
         ('d_b_eddy', ' - eddy'), ('d_b_eddy_vert', '   - vertical'),
         ('d_b_eddy_hor', '   - horizontal'), ('d_b_eddy_hor_direct', '   - horiz. direct'), ('d_b_sum', 'sum'), ('d_b_rate', 'online tendency')]

with warnings.catch_warnings():
    warnings.simplefilter('ignore', RuntimeWarning)
    bbl_mean = {key: np.nanmean(np.where(in_bbl, G[key], np.nan), axis=1) for key, _ in terms}
    table = {name: {key: stat(bbl_mean[key][sel]) for key, _ in terms} for name, sel in regions.items()}

print('BBL-mean dN2/dt terms [s-3] (%s over the columns of each region)' % STAT)
print('{:<24s}'.format('') + ''.join('{:>17s}'.format(label) for _, label in terms))
for name, row in table.items():
    print('{:<24s}'.format(name) + ''.join('{:>17.2e}'.format(row[key]) for key, _ in terms))

In [ ]:
# --- Instability diagnostics averaged over the BBL (0 < hab <= h_BBL), per region ------------------
def bbl_mean(field):
    with warnings.catch_warnings():
        warnings.simplefilter('ignore', RuntimeWarning)
        return np.nanmean(np.where(in_bbl, field, np.nan), axis=1)

def region_stat(values, sel, use_median=False):
    with warnings.catch_warnings():
        warnings.simplefilter('ignore', RuntimeWarning)
        return (np.nanmedian if use_median else stat)(values[sel])

# BBL deformation radius over the grid spacing: L_d/dx = N_BBL h_BBL / (f dx)   (needs N2_bbl > 0)
f_cols = simul.f.reshape(-1)[cols]
dx_cols = 1. / simul.pm.reshape(-1)[cols]
with np.errstate(invalid='ignore', divide='ignore'):
    Ld_dx = np.where(N2_bbl > 0, np.sqrt(np.where(N2_bbl > 0, N2_bbl, 1.)) * hbbl / f_cols / dx_cols, np.nan)

cols_mean = [('wpbp', "w'b'  [m2 s-3]"), ('C_hrs', 'HRS  [m2 s-3]'), ('C_vrs', 'VRS  [m2 s-3]'),
             ('P_h', 'P_h  [m2 s-5]'), ('P_v', 'P_v  [m2 s-5]'), ('P_tot', 'P_tot [m2 s-5]')]
cols_med = [('sigE', 'Eady [1/day]'), ('Ri', 'Ri'), ('sbur', 'slope ratio'), ('Ro', 'Ro')]
cols_mean = [(k, l) for k, l in cols_mean if k in H]

bm = {k: bbl_mean(H[k]) for k, _ in cols_mean + cols_med}
bm_cmp, bm_adi = bbl_mean(H['wpbp_cmp']), bbl_mean(H['wpbp_adiab'])

head = '{:<24s}'.format('') + ''.join('{:>14s}'.format(l) for _, l in cols_mean) \
       + '{:>10s}'.format("R_adiab") + ''.join('{:>13s}'.format(l) for _, l in cols_med) + '{:>9s}'.format('Ld/dx')
print('BBL-mean instability diagnostics (%s of the columns of each region; ratios: median)' % STAT)
print('HRS/VRS: barotropic / vertical-shear EKE sources; P_h, P_v: horizontal / vertical eddy APE production;')
print('R_adiab = w\'b\' / (along-isopycnal prediction); slope ratio = bottom slope / isopycnal slope')
print(head)
for name, sel in regions.items():
    row = '{:<24s}'.format(name)
    row += ''.join('{:>14.2e}'.format(region_stat(bm[k], sel)) for k, _ in cols_mean)
    row += '{:>10.2f}'.format(region_stat(bm_cmp, sel) / region_stat(bm_adi, sel))
    row += ''.join('{:>13.2e}'.format(region_stat(bm[k], sel, True)) for k, _ in cols_med)
    row += '{:>9.2f}'.format(region_stat(Ld_dx, sel, True))
    print(row)

In [ ]:
# --- Figures: one column per region, profiles against height above bottom ------------------------
LBL_ADV_MEAN = r'$-\overline{\mathbf{u}} \cdot \overline{\mathbf{\nabla} b}$'
LBL_ADV_EDDY = r'$-\overline{\mathbf{u^\prime} \cdot \mathbf{\nabla} b^\prime}$'
names = list(regions)


def plot_rows(rows, tag):
    """One figure, one column per region; each row is
    (x label, [(field, legend label, colour, line style)], options) with options 'xlim' and/or 'log'."""
    fig, axes = plt.subplots(len(rows), len(names), figsize=(3.3 * len(names), 3.6 * len(rows)),
                             sharey=True, squeeze=False)
    for r, (xlabel, curves, opts) in enumerate(rows):
        for c, name in enumerate(names):
            ax = axes[r, c]
            for key, label, color, style in curves:
                if key not in prof:                                # e.g. eddy-file term not available
                    continue
                if opts.get('log'):
                    ax.semilogx(prof[key][c], hab_grid, style, color=color, linewidth=2, label=label,
                                nonpositive='mask')
                else:
                    ax.plot(prof[key][c], hab_grid, style, color=color, linewidth=2, label=label)
            if opts.get('log'):
                ax.xaxis.set_minor_formatter(NullFormatter())      # avoid overlapping minor tick labels
            if opts.get('symlog'):
                ax.set_xscale('symlog', linthresh=1.)              # linear for |x| < 1, logarithmic beyond: shows Ri < 0 (N^2 < 0)
            if opts.get('vline') is not None:
                ax.axvline(opts['vline'], color='gray', linewidth=0.8)   # reference value (e.g. 1)
            elif not opts.get('log'):
                ax.axvline(0, color='gray', linewidth=0.8)         # zero line
            if opts.get('xlim') is not None:
                ax.set_xlim(opts['xlim'])
            ax.set_xlabel(xlabel)
            if r == 0:
                ax.set_title('%s\n(%d columns)' % (name, regions[name].sum()), fontsize=10)
            if c == 0:
                ax.set_ylabel('height above bottom [m]')
                ax.legend(fontsize=8)
    axes[0, 0].set_ylim(0, HAB_PLOT)                               # shared y axis
    fig.suptitle('%s profiles on hab - %s%s' % (STAT, simul.simul, '' if BMASK is None else '  (all budget terms: |b_adv| > %g masked)' % BMASK), fontsize=11)
    fig.tight_layout()
    plt.savefig(fifig + tag + simul.simul + '.png', dpi=200, bbox_inches='tight')


# 1) buoyancy budget | diffusivities | N^2
plot_rows([('buoyancy budget [m s$^{-3}$]',
            [('b_rhs', r'$b_{rhs}$', 'b', '-'), ('b_adv', r'$b_{adv}$', 'r', '-'),
             ('b_adv_mean', LBL_ADV_MEAN, 'y', '-'), ('b_eddy', LBL_ADV_EDDY, 'c', '-')],
            dict(xlim=(-1.5e-10, 1.5e-10))),
           ('diffusivity [m$^2$ s$^{-1}$]',
            [('K_eff', r'$K_{eff}$', 'k', '--'), ('K_KPP', r'$K_{KPP}$', 'k', '-')],
            dict(log=True)),
           ('$N^2$ [s$^{-2}$]',
            [('N2', r'$N^2$', 'k', '-')],
            dict(xlim=(-1e-6, 2e-6)))],
          'buoyancy_profiles_hab_')

# 2) stratification budget: vertical derivative of the buoyancy budget, dN2/dt = d(b_adv + b_rhs)/dz
plot_rows([('stratification budget [s$^{-3}$]',
            [('d_b_rhs', r'$\partial_z b_{rhs}$ (mixing)', 'b', '-'),
             ('d_b_adv', r'$\partial_z b_{adv}$', 'r', '-'),
             ('d_b_adv_mean', r'$\partial_z$ ' + LBL_ADV_MEAN, 'y', '-'),
             ('d_b_eddy', r'$\partial_z$ ' + LBL_ADV_EDDY, 'c', '-'),
             ('d_b_sum', r'adv + mixing', 'k', '--'),
             ('d_b_rate', r'$\partial_z b_{rate}$ (online)', 'gray', '-')],
            dict(xlim=STRAT_XLIM))],
          'stratification_budget_hab_')

# 3) eddy advection split into vertical and horizontal parts: buoyancy budget | stratification budget | flux
LBL_EDDY_VERT = r'vertical  $-\partial_z \overline{w^\prime b^\prime}$'
LBL_EDDY_HOR = 'horizontal (total - vertical)'
LBL_EDDY_HOR_D = r'horizontal, direct  $-\nabla_h\cdot\overline{\mathbf{u}_h^\prime b^\prime}$'
LBL_EDDY_DIR = r'eddy (total), from fluxes: vert + direct horiz.'
plot_rows([('buoyancy budget [m s$^{-3}$]',
            [('b_eddy', 'eddy (total) ' + LBL_ADV_EDDY, 'c', '-'),
             ('b_eddy_vert', LBL_EDDY_VERT, 'm', '-'),
             ('b_eddy_hor', LBL_EDDY_HOR, 'tab:orange', '-'),
             ('b_eddy_hor_direct', LBL_EDDY_HOR_D, 'tab:orange', '--'),
             ('b_eddy_direct', LBL_EDDY_DIR, 'c', '--')],
            dict(xlim=(-1.5e-10, 1.5e-10))),
           ('stratification budget [s$^{-3}$]',
            [('d_b_eddy', r'$\partial_z$ eddy (total)', 'c', '-'),
             ('d_b_eddy_vert', r'$\partial_z$ vertical', 'm', '-'),
             ('d_b_eddy_hor', r'$\partial_z$ horizontal', 'tab:orange', '-'),
             ('d_b_eddy_hor_direct', r'$\partial_z$ horizontal, direct', 'tab:orange', '--'),
             ('d_b_eddy_direct', r'$\partial_z$ eddy from fluxes', 'c', '--')],
            dict(xlim=STRAT_XLIM)),
           (r"$\overline{w^\prime b^\prime}$ [m$^2$ s$^{-3}$]",
            [('wpbp', r"$\overline{w^\prime b^\prime}$", 'm', '-')],
            dict())],
          'eddy_advection_split_hab_')

# 4) energetics: EKE sources | eddy APE production | along-isopycnal test
plot_rows([('EKE sources [m$^2$ s$^{-3}$]',
            [('wpbp', r"baroclinic $\overline{w^\prime b^\prime}$", 'm', '-'),
             ('C_hrs', 'barotropic (horiz. Reynolds stress)', 'g', '-'),
             ('C_vrs', 'vertical shear production', 'saddlebrown', '-')],
            dict()),
           ('eddy APE production [m$^2$ s$^{-5}$]',
            [('P_h', r"horizontal $-\overline{\mathbf{u}_h^\prime b^\prime}\cdot\nabla_h b$", 'tab:red', '-'),
             ('P_v', r"vertical $-\overline{w^\prime b^\prime} N^2$", 'tab:blue', '-'),
             ('P_tot', 'sum (0 if along isopycnals)', 'k', '--')],
            dict()),
           (r"$\overline{w^\prime b^\prime}$ / along-isopycnal value",
            [('R_adiab', 'ratio of region statistics', 'k', '-')],
            dict(xlim=(-1, 3), vline=1))],
          'eddy_energetics_hab_')

# 5) mean-state criteria for baroclinic instability (medians)
plot_rows([(r'Eady growth rate $0.31 M^2/N$ [day$^{-1}$]', [('sigE', 'median', 'k', '-')], dict()),
           (r'balanced Richardson number $N^2 f^2 / M^4$', [('Ri', 'median', 'k', '-')], dict(symlog=True, vline=1)),
           ('bottom slope / isopycnal slope', [('sbur', 'median', 'k', '-')], dict(log=True, vline=1)),
           (r'mean-flow Rossby number $\zeta/f$', [('Ro', 'median', 'k', '-')], dict())],
          'instability_criteria_hab_')

In [ ]:
# --- Summary figure (paper style): map of the regions | w profiles | buoyancy budget --------------------------------
from matplotlib.colors import ListedColormap
from matplotlib.gridspec import GridSpec
from matplotlib.lines import Line2D

# Letters of the sub-domains <-> regions (order of the `regions` dictionary)
LETTERS = [('A', names[2]), ('B', names[1]), ('C', names[3]), ('D', names[4]), ('All grid', names[0])]
SUB_COLORS = ['orange', 'purple', 'red', 'blue']
SUB_LABELS = [r'$w_{bot}>0$, $N^2_{BBL}>0$', r'$w_{bot}<0$, $N^2_{BBL}>0$',
              r'$w_{bot}<0$, $N^2_{BBL}<0$', r'$w_{bot}>0$, $N^2_{BBL}<0$']
FS = 15                                                           # base font size
LEG_FS = FS + 3                                                   # font size of the legends
LEG_SHIFT = -0.22   # shift of the subdomain legend to the left, in fractions of its axes width (0 = left edge of the legend axes; more negative = further left)
F_HEIGHT_FRAC = 0.85    # height of panel f (w profiles, all grid) relative to its full height (panel k keeps its height)
# BMASK (config cell): all budget terms are masked at the source where |b_adv| > BMASK
SHOW_EDDY_DIRECT = False  # also draw the eddy advection computed from the fluxes only (dashed)
hbbl_kpp = var('hbbl', simul).data.reshape(-1)[cols]              # KPP (parameterised mixing) BBL height

# Map of the sub-domains (columns not selected stay white)
region_id = np.full(nx * ny, np.nan)
for k, (_, rname) in enumerate(LETTERS[:4]):
    region_id[cols[regions[rname]]] = k
region_id = region_id.reshape(nx, ny)
xkm = np.cumsum(1. / simul.pm[:, ny // 2]) / 1e3                  # distance along xi / eta [km]
ykm = np.cumsum(1. / simul.pn[nx // 2, :]) / 1e3

with plt.rc_context({'font.size': FS, 'axes.titlesize': FS + 3, 'axes.labelsize': FS}):
    fig = plt.figure(figsize=(23, 20))
    gs = GridSpec(4, 6, height_ratios=[1.1, 2.0, 0.15, 2.95], width_ratios=[1, 1, 1, 1, 0.3, 1.8],   # column 4: spacer
                  hspace=0.32, wspace=0.22, figure=fig)

    # a) map -------------------------------------------------------------------------------
    ax = fig.add_subplot(gs[0, 0:2])
    ax.pcolormesh(xkm, ykm, region_id.T, cmap=ListedColormap(SUB_COLORS), vmin=-0.5, vmax=3.5, shading='auto')
    ax.contour(xkm, ykm, np.where(simul.mask > 0, simul.topo, np.nan).T,
               levels=np.arange(250., 6000., 250.), colors='k', linewidths=0.8)
    ax.set_aspect('equal', adjustable='box', anchor='NW')            # true distances: 1 km in xi = 1 km in eta
    ax.set_xlabel(r'km in $\xi$-direction'); ax.set_ylabel(r'km in $\eta$-direction')
    ax.set_title('a)')
    axl = fig.add_subplot(gs[0, 2:4]); axl.axis('off')
    handles = [Line2D([], [], marker='D', ls='', color=SUB_COLORS[k], markersize=9,
                      label='Subdomain %s (%s): %.2f %% of the grid'
                            % (LETTERS[k][0], SUB_LABELS[k], 100. * regions[LETTERS[k][1]].sum() / ncol))
               for k in range(4)]
    axl.legend(handles=handles, loc='center left', bbox_to_anchor=(LEG_SHIFT, 0.5), fontsize=LEG_FS, frameon=True)

    # b-f) w profiles: mean (solid), 10th and 90th percentiles (dashed), h_BBL (horizontal lines) ---
    # g-k) buoyancy budget terms [1e-11 m s-3]
    BUDGET = [('b_rhs', r'$b_{rhs}$ (mixing)', 'darkblue', '-'), ('b_adv', r'$b_{adv}$ (total advection)', 'red', '-'),
              ('b_adv_mean', r'$b^{mean}_{adv}$', 'lightgreen', '-'), ('b_eddy', r'$b^{eddy}_{adv}$', 'deepskyblue', '-')]
    if SHOW_EDDY_DIRECT:
        BUDGET.append(('b_eddy_direct', r'$b^{eddy}_{adv}$ (from fluxes)', 'deepskyblue', '--'))
    BUDGET = [b for b in BUDGET if b[0] in prof]

    def masked_stat(key, sel):
        """Region statistic of a budget term."""
        return stat(H[key][sel], axis=0)           # H terms are already masked where |b_adv| > BMASK
    hsel = hab_grid <= HAB_PLOT
    for c, (letter, rname) in enumerate(LETTERS):
        sel, ic = regions[rname], names.index(rname)
        if c < 4:
            axw = fig.add_subplot(gs[1, c]); axb = fig.add_subplot(gs[3, c])
        else:                                                     # all grid: two stacked panels over the full figure height
            axw = fig.add_subplot(gs[0:2, 5]); axb = fig.add_subplot(gs[2:4, 5])
        title = ('%s) Subdomain %s' % ('bcdef'[c], letter)) if c < 4 else 'f) All grid'
        tcol = SUB_COLORS[c] if c < 4 else 'k'                   # same colour as the subdomain in the map (a)
        axw.set_title(title, color=tcol)
        axb.set_title(('%s) Subdomain %s' % ('ghij'[c], letter)) if c < 4 else 'k) All grid', color=tcol)
        if sel.sum() > 0:
            with warnings.catch_warnings():
                warnings.simplefilter('ignore', RuntimeWarning)
                p10, p90 = np.nanpercentile(H['w'][sel], [10, 90], axis=0)
                hb_cur, hb_kpp = np.nanmean(hbbl[sel]), np.nanmean(hbbl_kpp[sel])
            axw.plot(prof['w'][ic], hab_grid, 'k-', lw=3)
            axw.plot(p10, hab_grid, 'k--', lw=2); axw.plot(p90, hab_grid, 'k--', lw=2)
            bprof = {}
            for key, label, color, style in BUDGET:
                bprof[key] = masked_stat(key, sel) * 1e11
                axb.plot(bprof[key], hab_grid, style, color=color, linewidth=3)
            # x range from all plotted terms below HAB_PLOT
            ref = np.nanmax(np.abs(np.concatenate([bprof[k][hsel] for k in bprof])))
            if np.isfinite(ref) and ref > 0:
                axb.set_xlim(-1.15 * ref, 1.15 * ref)
            for a_ in (axw, axb):
                a_.axhline(hb_cur, color='k', ls='--', lw=1.8, alpha=0.8)    # curvature method
                a_.axhline(hb_kpp, color='r', ls='--', lw=1.8, alpha=0.8)    # KPP mixing
        else:
            for a_ in (axw, axb):
                a_.text(0.5, 0.5, 'no columns', transform=a_.transAxes, ha='center')
        axw.set_xlabel(r'w [m day$^{-1}$]')
        axb.set_xlabel(r'b [10$^{-11}$ m s$^{-3}$]')
        for a_ in (axw, axb):
            a_.set_ylim(0, HAB_PLOT); a_.grid(True, alpha=0.4)
            a_.axvline(0, color='gray', lw=1.5)
            if c == 0 or c == 4:
                a_.set_ylabel('hab [m]')
            else:
                a_.set_yticklabels([])

    # legend of the budget panels
    axg = fig.add_subplot(gs[2, 0:4]); axg.axis('off')
    hl = [Line2D([], [], color=col, ls=st, lw=3, label=lab) for key, lab, col, st in BUDGET]
    hl += [Line2D([], [], color='k', ls='--', lw=1.8, label=r'$h_{BBL}$ curvature'),
           Line2D([], [], color='r', ls='--', lw=1.8, label=r'$h_{BBL}$ KPP')]
    axg.legend(handles=hl, loc='center', ncol=3, fontsize=LEG_FS)
    # panel f (last `axw` of the loop above) a little less high than panel k: keep its bottom, lower its top
    pf = axw.get_position()
    axw.set_position([pf.x0, pf.y0, pf.width, pf.height * F_HEIGHT_FRAC])
    plt.savefig(fifig + 'figure6_corrected_' + simul.simul + '.png', dpi=200, bbox_inches='tight')


In [ ]:
# --- Summary figure (paper style) for the STRATIFICATION budget: map | 4 subdomains + all grid -----------------------
# Same regions/colours as the buoyancy summary figure above; d(term)/dz [1e-13 s-3], no horizontal/vertical eddy split.
SBUDGET = [('d_b_rhs', r'$\partial_z b_{rhs}$ (mixing)', 'darkblue', '-'),
           ('d_b_adv', r'$\partial_z b_{adv}$ (total advection)', 'red', '-'),
           ('d_b_adv_mean', r'$\partial_z b^{mean}_{adv}$', 'lightgreen', '-'),
           ('d_b_eddy', r'$\partial_z b^{eddy}_{adv}$', 'deepskyblue', '-'),
           ('d_b_rate', r'online tendency of $N^2$', 'k', ':')]
SBUDGET = [s for s in SBUDGET if s[0] in G]

with plt.rc_context({'font.size': FS, 'axes.titlesize': FS + 3, 'axes.labelsize': FS}):
    fig = plt.figure(figsize=(26, 13))
    gs = GridSpec(2, 6, height_ratios=[1.0, 2.2], width_ratios=[1, 1, 1, 1, 0.3, 1.8], hspace=0.3, wspace=0.22, figure=fig)

    # a) map of the subdomains
    ax = fig.add_subplot(gs[0, 0:2])
    ax.pcolormesh(xkm, ykm, region_id.T, cmap=ListedColormap(SUB_COLORS), vmin=-0.5, vmax=3.5, shading='auto')
    ax.contour(xkm, ykm, np.where(simul.mask > 0, simul.topo, np.nan).T,
               levels=np.arange(250., 6000., 250.), colors='k', linewidths=0.8)
    ax.set_aspect('equal', adjustable='box', anchor='NW')
    ax.set_xlabel(r'km in $\xi$-direction'); ax.set_ylabel(r'km in $\eta$-direction'); ax.set_title('a)')
    axl = fig.add_subplot(gs[0, 2:6]); axl.axis('off')
    handles = [Line2D([], [], marker='D', ls='', color=SUB_COLORS[k], markersize=9,
                      label='Subdomain %s (%s): %.2f %% of the grid'
                            % (LETTERS[k][0], SUB_LABELS[k], 100. * regions[LETTERS[k][1]].sum() / ncol))
               for k in range(4)]
    handles += [Line2D([], [], color=col, ls=st, lw=3, label=lab) for key, lab, col, st in SBUDGET]
    handles += [Line2D([], [], color='k', ls='--', lw=1.8, label=r'$h_{BBL}$ curvature'),
                Line2D([], [], color='r', ls='--', lw=1.8, label=r'$h_{BBL}$ KPP')]
    axl.legend(handles=handles, loc='center left', ncol=2, fontsize=FS - 1, frameon=True)

    # b-f) stratification budget [1e-13 s-3]
    hsel = hab_grid <= HAB_PLOT
    for c, (letter, rname) in enumerate(LETTERS):
        sel = regions[rname]
        axb = fig.add_subplot(gs[1, c if c < 4 else 5])
        tcol = SUB_COLORS[c] if c < 4 else 'k'
        axb.set_title(('%s) Subdomain %s' % ('bcde'[c], letter)) if c < 4 else 'f) All grid', color=tcol)
        if sel.sum() > 0:
            with warnings.catch_warnings():
                warnings.simplefilter('ignore', RuntimeWarning)
                hb_cur, hb_kpp = np.nanmean(hbbl[sel]), np.nanmean(hbbl_kpp[sel])
                sprof = {}
                for key, label, color, style in SBUDGET:
                    sprof[key] = stat(G[key][sel], axis=0) * 1e13
                    axb.plot(sprof[key], hab_grid, style, color=color, linewidth=3)
                ref = np.nanmax(np.abs(np.concatenate([sprof[k][hsel] for k in sprof])))
            if STRAT_XLIM is not None:
                axb.set_xlim(np.array(STRAT_XLIM) * 1e13)
            elif np.isfinite(ref) and ref > 0:
                axb.set_xlim(-1.15 * ref, 1.15 * ref)
            axb.axhline(hb_cur, color='k', ls='--', lw=1.8, alpha=0.8)
            axb.axhline(hb_kpp, color='r', ls='--', lw=1.8, alpha=0.8)
        else:
            axb.text(0.5, 0.5, 'no columns', transform=axb.transAxes, ha='center')
        axb.set_xlabel(r'$\partial N^2/\partial t$ terms [10$^{-13}$ s$^{-3}$]')
        axb.set_ylim(0, HAB_PLOT); axb.grid(True, alpha=0.4); axb.axvline(0, color='gray', lw=1.5)
        if c in (0, 4):
            axb.set_ylabel('hab [m]')
        else:
            axb.set_yticklabels([])
    plt.savefig(fifig + 'summary_stratification_regions_hab_' + simul.simul + '.png', dpi=200, bbox_inches='tight')


In [ ]:
# --- Buoyancy balance of subdomains A and B: (advection + mixing) and (advection split into mean / eddy) -------------
# Panels 1-2: A, B with b_adv and b_rhs only; panels 3-4: A, B with b_adv, b_rhs, b_adv_mean and b_eddy. Same x range everywhere.
AB = [(0, 'A'), (1, 'B')]                                          # indices in LETTERS / SUB_COLORS
TERMS_2 = [('b_rhs', r'$b_{rhs}$ (mixing)', 'darkblue', '-'), ('b_adv', r'$b_{adv}$ (total advection)', 'red', '-')]
TERMS_4 = TERMS_2 + [('b_adv_mean', r'$b^{mean}_{adv}$', 'limegreen', '-'), ('b_eddy', r'$b^{eddy}_{adv}$', 'deepskyblue', '-')]
hsel = hab_grid <= HAB_PLOT

curves = {}
with warnings.catch_warnings():
    warnings.simplefilter('ignore', RuntimeWarning)
    for ia, _ in AB:
        sel = regions[LETTERS[ia][1]]
        for key, *_ in TERMS_4:
            curves[(ia, key)] = stat(H[key][sel], axis=0) * 1e11 if sel.sum() > 0 else np.full(hab_grid.shape, np.nan)
ref = np.nanmax(np.abs(np.concatenate([c[hsel] for c in curves.values()])))   # common x range for the 4 panels

with plt.rc_context({'font.size': FS, 'axes.titlesize': FS + 2, 'axes.labelsize': FS}):
    fig, axs = plt.subplots(1, 4, figsize=(24, 8), sharex=True, sharey=True, constrained_layout=True)
    for n, (terms, tag) in enumerate([(TERMS_2, 'adv + mixing'), (TERMS_2, 'adv + mixing'), (TERMS_4, 'adv: mean / eddy'), (TERMS_4, 'adv: mean / eddy')]):
        ia = AB[n % 2][0]
        ax = axs[n]
        sel = regions[LETTERS[ia][1]]
        for key, label, color, style in terms:
            ax.plot(curves[(ia, key)], hab_grid, style, color=color, linewidth=3, label=label)
        if sel.sum() > 0:
            with warnings.catch_warnings():
                warnings.simplefilter('ignore', RuntimeWarning)
                ax.axhline(np.nanmean(hbbl[sel]), color='k', ls='--', lw=1.8, alpha=0.8, label=r'$h_{BBL}$ curvature')
                ax.axhline(np.nanmean(hbbl_kpp[sel]), color='r', ls='--', lw=1.8, alpha=0.8, label=r'$h_{BBL}$ KPP')
        ax.set_title('%s) Subdomain %s' % ('abcd'[n], LETTERS[ia][0]), color=SUB_COLORS[ia])
        ax.set_xlabel(r'b [10$^{-11}$ m s$^{-3}$]'); ax.grid(True, alpha=0.4); ax.axvline(0, color='gray', lw=1.5)
        if n == 0:
            ax.set_ylabel('hab [m]')
    axs[0].set_xlim(-1.15 * ref, 1.15 * ref); axs[0].set_ylim(0, HAB_PLOT)
    axs[0].legend(loc='best', fontsize=FS - 3); axs[2].legend(loc='best', fontsize=FS - 3)
    fig.suptitle('Buoyancy balance of subdomains A and B (same x range)', fontsize=FS + 3)
    plt.savefig(fifig + 'buoyancy_balance_A_B_hab_' + simul.simul + '.png', dpi=200, bbox_inches='tight')


In [ ]:
# --- Stratification budget of the whole domain, 3 panels with increasing detail (same x range) -------------------------
# 1: advection + mixing | 2: + mean / eddy advection | 3: + vertical / horizontal eddy advection.   d(term)/dz [1e-13 s-3]
S1 = [('d_b_rhs', r'$\partial_z b_{rhs}$ (mixing)', 'darkblue', '-'), ('d_b_adv', r'$\partial_z b_{adv}$ (total advection)', 'red', '-')]
S2 = S1 + [('d_b_adv_mean', r'$\partial_z b^{mean}_{adv}$', 'limegreen', '-'), ('d_b_eddy', r'$\partial_z b^{eddy}_{adv}$', 'deepskyblue', '-')]
S3 = S2 + [('d_b_eddy_vert', r'$\partial_z b^{eddy}_{adv}$ vertical', 'deepskyblue', '--'),
           ('d_b_eddy_hor', r'$\partial_z b^{eddy}_{adv}$ horizontal', 'navy', ':')]
S3 = [s for s in S3 if s[0] in G]
sel_all = regions[names[0]]
hsel = hab_grid <= HAB_PLOT
with warnings.catch_warnings():
    warnings.simplefilter('ignore', RuntimeWarning)
    sc = {k: stat(G[k][sel_all], axis=0) * 1e13 for k, *_ in S3}
    hb_cur, hb_kpp = np.nanmean(hbbl[sel_all]), np.nanmean(hbbl_kpp[sel_all])
ref = np.nanmax(np.abs(np.concatenate([c[hsel] for c in sc.values()])))      # common range for the 3 panels

with plt.rc_context({'font.size': FS, 'axes.titlesize': FS + 2, 'axes.labelsize': FS}):
    fig, axs = plt.subplots(1, 3, figsize=(20, 8), sharex=True, sharey=True, constrained_layout=True)
    for n, (terms, title) in enumerate([(S1, 'a) advection + mixing'), (S2, 'b) + mean / eddy advection'),
                                        (S3, 'c) + vertical / horizontal eddy advection')]):
        ax = axs[n]
        for key, label, color, style in terms:
            if key in sc:
                ax.plot(sc[key], hab_grid, style, color=color, linewidth=3, label=label)
        ax.axhline(hb_cur, color='k', ls='--', lw=1.8, alpha=0.8, label=r'$h_{BBL}$ curvature')
        ax.axhline(hb_kpp, color='r', ls='--', lw=1.8, alpha=0.8, label=r'$h_{BBL}$ KPP')
        ax.set_title(title); ax.set_xlabel(r'$\partial N^2/\partial t$ terms [10$^{-13}$ s$^{-3}$]')
        ax.grid(True, alpha=0.4); ax.axvline(0, color='gray', lw=1.5)
        ax.legend(loc='best', fontsize=FS - 4)
    axs[0].set_ylabel('hab [m]')
    axs[0].set_ylim(0, HAB_PLOT)
    axs[0].set_xlim(-1.15 * ref, 1.15 * ref) if STRAT_XLIM is None else axs[0].set_xlim(np.array(STRAT_XLIM) * 1e13)
    fig.suptitle('Stratification budget, all grid (same x range)', fontsize=FS + 3)
    plt.savefig(fifig + 'stratification_budget_all_3panels_hab_' + simul.simul + '.png', dpi=200, bbox_inches='tight')


In [ ]:
# --- Figure for the whole selected domain ("All"): w'b', buoyancy budget, stratification budget ----------------------
# Eddy advection split into its vertical part -d(w'b')/dz and the rest (horizontal part); the 'All' region is the first one.
iall = 0
sel_all = regions[names[iall]]
hb_cur, hb_kpp = np.nanmean(hbbl[sel_all]), np.nanmean(hbbl_kpp[sel_all])
hsel = hab_grid <= HAB_PLOT
FSA = 14

B_TERMS = [('b_rhs', r'$b_{rhs}$ (mixing)', 'darkblue', '-'), ('b_adv', r'$b_{adv}$ (total advection)', 'red', '-'),
           ('b_adv_mean', r'$b^{mean}_{adv}$', 'limegreen', '-'), ('b_eddy', r'$b^{eddy}_{adv}$', 'deepskyblue', '-'),
           ('b_eddy_vert', r'$b^{eddy}_{adv}$ vertical: $-\partial_z\overline{w^\prime b^\prime}$', 'deepskyblue', '--'),
           ('b_eddy_hor', r'$b^{eddy}_{adv}$ horizontal (total $-$ vertical)', 'navy', ':')]
S_TERMS = [('d_b_rhs', r'$\partial_z b_{rhs}$ (mixing)', 'darkblue', '-'), ('d_b_adv', r'$\partial_z b_{adv}$', 'red', '-'),
           ('d_b_adv_mean', r'$\partial_z b^{mean}_{adv}$', 'limegreen', '-'), ('d_b_eddy', r'$\partial_z b^{eddy}_{adv}$', 'deepskyblue', '-'),
           ('d_b_eddy_vert', r'$\partial_z b^{eddy}_{adv}$ vertical', 'deepskyblue', '--'),
           ('d_b_eddy_hor', r'$\partial_z b^{eddy}_{adv}$ horizontal', 'navy', ':'),
           ('d_b_rate', 'online tendency of $N^2$', 'k', '-')]


def stat_all(field, mask=None):
    f = field[sel_all]
    if mask is not None:
        f = np.where(np.abs(f) > mask, np.nan, f)
    with warnings.catch_warnings():
        warnings.simplefilter('ignore', RuntimeWarning)
        return stat(f, axis=0)


def draw_terms(ax, terms, source, scale, mask=None, lw=2.):
    curves = {}
    for key, label, color, style in terms:
        if key in source:
            curves[key] = stat_all(source[key], mask) * scale
            ax.plot(curves[key], hab_grid, style, color=color, linewidth=lw, label=label)
    ref = np.nanmax(np.abs(np.concatenate([c[hsel] for c in curves.values()])))
    return ref


with plt.rc_context({'font.size': FSA, 'axes.titlesize': FSA + 2, 'axes.labelsize': FSA}):
    fig, axs = plt.subplots(1, 3, figsize=(24, 9), sharey=True, constrained_layout=True)

    # a) vertical eddy buoyancy flux w'b' : mean, 25-75 % and 10-90 % ranges over the columns
    ax = axs[0]
    wb = H['wpbp'][sel_all]
    with warnings.catch_warnings():
        warnings.simplefilter('ignore', RuntimeWarning)
        q10, q25, q75, q90 = np.nanpercentile(wb, [10, 25, 75, 90], axis=0)
    ax.fill_betweenx(hab_grid, q10, q90, color='0.88', label='10-90 %')
    ax.fill_betweenx(hab_grid, q25, q75, color='0.7', label='25-75 %')
    ax.plot(stat_all(H['wpbp']), hab_grid, 'k-', linewidth=2.5, label=r"%s of $\overline{w^\prime b^\prime}$" % STAT)
    if 'wpbp_adiab' in H:
        ax.plot(stat_all(H['wpbp_adiab']), hab_grid, 'm--', linewidth=1.5,
                label=r"along-isopycnal value $\overline{\mathbf{u}_h^\prime b^\prime}\cdot\nabla_h b/N^2$")
    ax.set_xlabel(r"$\overline{w^\prime b^\prime}$ [m$^2$ s$^{-3}$]"); ax.set_title("a) vertical eddy buoyancy flux")
    lim = np.nanmax(np.abs(np.concatenate([q25[hsel], q75[hsel]])))
    if np.isfinite(lim) and lim > 0:
        ax.set_xlim(-3 * lim, 3 * lim)

    # b) buoyancy budget [1e-11 m s-3] (points with |b_adv| > BMASK already masked)
    ax = axs[1]
    ref = draw_terms(ax, B_TERMS, H, 1e11)
    ax.set_xlabel(r'b [10$^{-11}$ m s$^{-3}$]'); ax.set_title('b) buoyancy budget')
    if np.isfinite(ref) and ref > 0:
        ax.set_xlim(-1.15 * ref, 1.15 * ref)

    # c) stratification budget [1e-13 s-3]
    ax = axs[2]
    ref = draw_terms(ax, S_TERMS, G, 1e13)
    ax.set_xlabel(r'$\partial N^2/\partial t$ terms [10$^{-13}$ s$^{-3}$]'); ax.set_title('c) stratification budget')
    if STRAT_XLIM is not None:
        ax.set_xlim(np.array(STRAT_XLIM) * 1e13)
    elif np.isfinite(ref) and ref > 0:
        ax.set_xlim(-1.15 * ref, 1.15 * ref)

    for k, ax in enumerate(axs):
        ax.axhline(hb_cur, color='k', ls='--', lw=0.9, alpha=0.8, label=r'$h_{BBL}$ curvature' if k else None)
        ax.axhline(hb_kpp, color='r', ls='--', lw=0.9, alpha=0.8, label=r'$h_{BBL}$ KPP' if k else None)
        ax.axvline(0, color='gray', lw=0.8); ax.grid(True, alpha=0.4)
        ax.set_ylim(0, HAB_PLOT); ax.legend(fontsize=FSA - 3, loc='best')
    axs[0].set_ylabel('height above bottom [m]')
    fig.suptitle('%s: %s (%d columns)' % (simul.simul, names[iall], sel_all.sum()), fontsize=FSA + 3)
    plt.savefig(fifig + 'all_domain_wpbp_budgets_hab_' + simul.simul + '.png', dpi=200, bbox_inches='tight')


In [ ]:
# --- Map of w'b' vertically integrated over the BBL (0 < hab <= h_BBL) -------------------------------------------------
# I = int_0^{h_BBL} w'b' dz  [m3 s-3] (trapezoid-free sum over the hab grid, spacing DH; hab = 0 has no data), and its BBL mean I / h_BBL.
with warnings.catch_warnings():
    warnings.simplefilter('ignore', RuntimeWarning)
    wb_bbl = np.where(in_bbl, H['wpbp'], np.nan)
    I_wb = np.nansum(wb_bbl, axis=1) * DH
    I_wb[np.isnan(wb_bbl).all(axis=1)] = np.nan
    M_wb = I_wb / hbbl                                              # BBL-mean w'b' [m2 s-3]

def to_map(v):
    m = np.full(nx * ny, np.nan); m[cols] = v
    return m.reshape(nx, ny)

maps = [(to_map(I_wb), r"$\int_0^{h_{BBL}}\overline{w^\prime b^\prime}\,dz$ [m$^3$ s$^{-3}$]"),
        (to_map(M_wb), r"BBL-mean $\overline{w^\prime b^\prime}$ [m$^2$ s$^{-3}$]"),
        (to_map(hbbl), r"$h_{BBL}$ (curvature) [m]")]
xkm = np.cumsum(1. / simul.pm[:, ny // 2]) / 1e3
ykm = np.cumsum(1. / simul.pn[nx // 2, :]) / 1e3
print("BBL-integrated w'b': mean %.2e, 1st/99th percentiles %.2e / %.2e m3 s-3"
      % (np.nanmean(I_wb), *np.nanpercentile(I_wb, [1, 99])))

fig, axs = plt.subplots(1, 3, figsize=(24, 7.5), sharex=True, sharey=True, constrained_layout=True)
for k, (ax, (field, label)) in enumerate(zip(axs, maps)):
    if k < 2:
        v = np.nanpercentile(np.abs(field), 98)
        pc = ax.pcolormesh(xkm, ykm, field.T, shading='auto', cmap='RdBu_r', vmin=-v, vmax=v)
    else:
        pc = ax.pcolormesh(xkm, ykm, field.T, shading='auto', cmap='viridis', vmin=0, vmax=HBBL_SEARCH_MAX)
    ax.contour(xkm, ykm, np.where(simul.mask > 0, simul.topo, np.nan).T,
               levels=np.arange(250., 6000., 250.), colors='k', linewidths=0.3)
    fig.colorbar(pc, ax=ax, extend='both' if k < 2 else 'max', label=label)
    ax.set_title('abc'[k] + ')')
    ax.set_xlabel(r'km in $\xi$-direction')
axs[0].set_ylabel(r'km in $\eta$-direction')
plt.savefig(fifig + 'map_wpbp_bbl_integrated_' + simul.simul + '.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# --- Columns where the BBL-mean w'b' is positive (baroclinic release) vs negative: all the budgets -------------------------
# Selection: BBL-mean w'b' (M_wb, from the map above) > 0.  Rows: that class | the complementary class (w'b' < 0).
# Columns: EKE sources | eddy APE production | eddy buoyancy fluxes (vertical, horizontal) | buoyancy budget | stratification budget.
WB_BMASK = None                                                 # no extra mask: terms are already masked where |b_adv| > BMASK
classes = [(r"BBL-mean $\overline{w^\prime b^\prime}>0$", np.isfinite(M_wb) & (M_wb > 0)),
           (r"BBL-mean $\overline{w^\prime b^\prime}<0$", np.isfinite(M_wb) & (M_wb < 0))]

# F_h (horizontal eddy buoyancy flux down the mean gradient, H['F_h']) and the topostrophy tau_cols are computed in the instability cell.

PAN = [  # (title, x label, scale, mask, [(source, key, label, colour, style)])
    ('EKE sources', r'[10$^{-9}$ m$^2$ s$^{-3}$]', 1e9, None,
     [(H, 'wpbp', r"baroclinic $\overline{w^\prime b^\prime}$", 'm', '-'),
      (H, 'C_hrs', 'barotropic (horiz. Reynolds stress)', 'g', '-'),
      (H, 'C_vrs', 'vertical shear production', 'saddlebrown', '-')]),
    ('eddy APE production', r'[10$^{-15}$ m$^2$ s$^{-5}$]', 1e15, None,
     [(H, 'P_h', r"horizontal $-\overline{\mathbf{u}_h^\prime b^\prime}\cdot\nabla_h b$", 'tab:red', '-'),
      (H, 'P_v', r"vertical $-\overline{w^\prime b^\prime}N^2$", 'tab:blue', '-'),
      (H, 'P_tot', 'sum', 'k', '--')]),
    ('eddy buoyancy fluxes', r'[10$^{-9}$ m$^2$ s$^{-3}$]', 1e9, None,
     [(H, 'wpbp_cmp', r"vertical flux $\overline{w^\prime b^\prime}$", 'm', '-'),
      (H, 'wpbp_adiab', r"horizontal flux projected on the vertical along isopycnals: $\overline{\mathbf{u}_h^\prime b^\prime}\cdot\nabla_h b\,/N^2$ (> 0 if baroclinic)", 'tab:orange', '-')]),
    ('buoyancy budget', r'[10$^{-11}$ m s$^{-3}$]', 1e11, WB_BMASK,
     [(H, 'b_rhs', r'$b_{rhs}$ (mixing)', 'darkblue', '-'), (H, 'b_adv', r'$b_{adv}$', 'red', '-'),
      (H, 'b_adv_mean', r'$b^{mean}_{adv}$', 'limegreen', '-'), (H, 'b_eddy', r'$b^{eddy}_{adv}$', 'deepskyblue', '-'),
      (H, 'b_eddy_vert', r'eddy vertical', 'deepskyblue', '--'), (H, 'b_eddy_hor', r'eddy horizontal', 'navy', ':')]),
    ('stratification budget', r'[10$^{-13}$ s$^{-3}$]', 1e13, None,
     [(G, 'd_b_rhs', r'$\partial_z b_{rhs}$', 'darkblue', '-'), (G, 'd_b_adv', r'$\partial_z b_{adv}$', 'red', '-'),
      (G, 'd_b_adv_mean', r'$\partial_z b^{mean}_{adv}$', 'limegreen', '-'), (G, 'd_b_eddy', r'$\partial_z b^{eddy}_{adv}$', 'deepskyblue', '-'),
      (G, 'd_b_eddy_vert', 'eddy vertical', 'deepskyblue', '--'), (G, 'd_b_eddy_hor', 'eddy horizontal', 'navy', ':')]),
]


def class_stat(field, sel, mask=None):
    f = field[sel]
    if mask is not None:
        f = np.where(np.abs(f) > mask, np.nan, f)
    with warnings.catch_warnings():
        warnings.simplefilter('ignore', RuntimeWarning)
        return stat(f, axis=0)


# Map of the two classes
xkm = np.cumsum(1. / simul.pm[:, ny // 2]) / 1e3
ykm = np.cumsum(1. / simul.pn[nx // 2, :]) / 1e3
cls_map = np.full(nx * ny, np.nan)
for k, (_, sel) in enumerate(classes):
    cls_map[cols[sel]] = k
fig, axm = plt.subplots(1, 3, figsize=(22, 5.8), sharex=True, sharey=True, constrained_layout=True)
topo_c = np.where(simul.mask > 0, simul.topo, np.nan).T
ax = axm[0]
ax.pcolormesh(xkm, ykm, cls_map.reshape(nx, ny).T, cmap=ListedColormap(['tab:red', 'lightsteelblue']), vmin=-0.5, vmax=1.5, shading='auto')
ax.set_title("a) red: BBL-mean $\\overline{w'b'}>0$ (%.1f %% of the columns); blue: $<0$ (%.1f %%)"
             % tuple(100. * sel.sum() / ncol for _, sel in classes), fontsize=11)
wmap = np.full(nx * ny, np.nan); wmap[cols] = wbot * 86400.
tmap = np.full(nx * ny, np.nan); tmap[cols] = tau_cols * 1e3
for ax, fld, ttl, lab in [(axm[1], wmap, 'b) bottom vertical velocity', r'$w_{bot}$ [m day$^{-1}$]'),
                          (axm[2], tmap, 'c) topostrophy of the bottom flow', r'$\tau$ [10$^{-3}$ m s$^{-1}$]')]:
    v = np.nanpercentile(np.abs(fld), 98)
    pc = ax.pcolormesh(xkm, ykm, fld.reshape(nx, ny).T, cmap='RdBu_r', vmin=-v, vmax=v, shading='auto')
    fig.colorbar(pc, ax=ax, extend='both', label=lab); ax.set_title(ttl, fontsize=11)
for ax in axm:
    ax.contour(xkm, ykm, topo_c, levels=np.arange(250., 6000., 250.), colors='k', linewidths=0.3)
    ax.set_xlabel(r'km in $\xi$-direction')
axm[0].set_ylabel(r'km in $\eta$-direction')
plt.savefig(fifig + 'map_wpbp_positive_region_' + simul.simul + '.png', dpi=150, bbox_inches='tight')
plt.show()

# Profiles
hsel = hab_grid <= HAB_PLOT
FSC = 13
with plt.rc_context({'font.size': FSC, 'axes.titlesize': FSC + 1, 'axes.labelsize': FSC}):
    NC = len(PAN) + 2                                                   # + w profile + topostrophy PDF
    fig, axs = plt.subplots(2, NC, figsize=(5.4 * NC, 11), constrained_layout=True)
    for r, (cname, sel) in enumerate(classes):
        hb = np.nanmean(hbbl[sel]); hbk = np.nanmean(hbbl_kpp[sel]) if 'hbbl_kpp' in globals() else np.nan
        for c, (title, xlabel, scale, mask, curves) in enumerate(PAN):
            ax = axs[r, c]; prof_ = []
            for source, key, label, color, style in curves:
                if key in source:
                    p = class_stat(source[key], sel, mask) * scale
                    prof_.append(p[hsel])
                    ax.plot(p, hab_grid, style, color=color, lw=2, label=label)
            if prof_:
                ref = np.nanmax(np.abs(np.concatenate(prof_)))
                if np.isfinite(ref) and ref > 0:
                    ax.set_xlim(-1.15 * ref, 1.15 * ref)
            ax.axhline(hb, color='k', ls='--', lw=0.9); ax.axhline(hbk, color='r', ls='--', lw=0.9)
            ax.axvline(0, color='gray', lw=0.8); ax.grid(True, alpha=0.4); ax.set_ylim(0, HAB_PLOT)
            ax.set_xlabel(xlabel)
            if r == 0:
                ax.set_title(title)
            if c > 0:
                ax.tick_params(labelleft=False)
            if c == 0:
                ax.set_ylabel('hab [m]')
                ax.text(0.03, 0.97, '%s\n%d columns' % (cname, sel.sum()), transform=ax.transAxes, va='top', fontsize=FSC - 1,
                        bbox=dict(fc='w', alpha=0.8, ec='none'))
            if r == 0:
                ax.legend(fontsize=FSC - 4, loc='lower right' if c in (3, 4) else 'best')
        # w profile on hab (mean, 10-90 % range) -------------------------------------------------------
        ax = axs[r, len(PAN)]
        with warnings.catch_warnings():
            warnings.simplefilter('ignore', RuntimeWarning)
            p10, p90 = np.nanpercentile(H['w'][sel], [10, 90], axis=0)
        ax.fill_betweenx(hab_grid, p10, p90, color='0.85', label='10-90 %')
        ax.plot(class_stat(H['w'], sel), hab_grid, 'k-', lw=2, label=r'%s $w$' % STAT)
        ax.axhline(hb, color='k', ls='--', lw=0.9); ax.axhline(hbk, color='r', ls='--', lw=0.9)
        ax.axvline(0, color='gray', lw=0.8); ax.grid(True, alpha=0.4); ax.set_ylim(0, HAB_PLOT)
        ax.tick_params(labelleft=False); ax.set_xlabel(r'$w$ [m day$^{-1}$]')
        if r == 0:
            ax.set_title('vertical velocity'); ax.legend(fontsize=FSC - 4, loc='best')
        # PDF of the topostrophy of the bottom flow ------------------------------------------------------
        ax = axs[r, len(PAN) + 1]
        tv = tau_cols * 1e3
        lim = np.nanpercentile(np.abs(tv[np.isfinite(tv)]), 98)
        bins = np.linspace(-lim, lim, 61)
        ax.hist(tv[sel & np.isfinite(tv)], bins=bins, density=True, color='tab:red' if r == 0 else 'lightsteelblue', alpha=0.85)
        ax.axvline(0, color='gray', lw=0.8)
        ax.axvline(np.nanmean(tv[sel]), color='k', lw=1.5, label='mean %.2g' % np.nanmean(tv[sel]))
        ax.set_xlabel(r'topostrophy $\tau$ [10$^{-3}$ m s$^{-1}$]'); ax.grid(True, alpha=0.4)
        ax.text(0.97, 0.97, r'$\tau>0$: %.0f %%' % (100. * np.nanmean(tv[sel] > 0)) + '\n' + r'mean $w_{bot}$: %.2g m day$^{-1}$' % (np.nanmean(wbot[sel]) * 86400.),
                transform=ax.transAxes, ha='right', va='top', fontsize=FSC - 2, bbox=dict(fc='w', alpha=0.8, ec='none'))
        if r == 0:
            ax.set_title('topostrophy (bottom flow)')
        ax.legend(fontsize=FSC - 4, loc='lower right')
    fig.suptitle(simul.simul + r"   (dashed lines: mean $h_{BBL}$, curvature in black, KPP in red)", fontsize=FSC + 2)
    plt.savefig(fifig + 'wpbp_positive_region_budgets_hab_' + simul.simul + '.png', dpi=200, bbox_inches='tight')

# BBL-mean numbers of the two classes
def bmean(f, sel):
    with warnings.catch_warnings():
        warnings.simplefilter('ignore', RuntimeWarning)
        return stat(np.nanmean(np.where(in_bbl[sel], f[sel], np.nan), axis=1))

rows = [('w (hab, mean)', H, 'w'), ('w\'b\'', H, 'wpbp'), ('F_h', H, 'F_h'), ('HRS', H, 'C_hrs'), ('VRS', H, 'C_vrs'), ('P_h', H, 'P_h'), ('P_v', H, 'P_v'),
        ('b_rhs', H, 'b_rhs'), ('b_adv', H, 'b_adv'), ('b_adv_mean', H, 'b_adv_mean'), ('b_eddy', H, 'b_eddy'),
        ('b_eddy_vert', H, 'b_eddy_vert'), ('b_eddy_hor', H, 'b_eddy_hor'),
        ('dz b_rhs', G, 'd_b_rhs'), ('dz b_adv', G, 'd_b_adv'), ('dz b_eddy', G, 'd_b_eddy'),
        ('dz b_eddy_vert', G, 'd_b_eddy_vert'), ('dz b_eddy_hor', G, 'd_b_eddy_hor')]
print('BBL-mean values (%s over the columns of each class)' % STAT)
print('{:<16s}'.format('') + ''.join('{:>18s}'.format(('w\'b\'>0' if k == 0 else 'w\'b\'<0') + ' (%d)' % sel.sum()) for k, (_, sel) in enumerate(classes)))
for label, src_, key in rows:
    if key in src_:
        print('{:<16s}'.format(label) + ''.join('{:>18.2e}'.format(bmean(src_[key], sel)) for _, sel in classes))

print('{:<16s}'.format('w_bot [m/day]') + ''.join('{:>18.2e}'.format(np.nanmean(wbot[sel]) * 86400.) for _, sel in classes))
print('{:<16s}'.format('topostrophy') + ''.join('{:>18.2e}'.format(np.nanmean(tau_cols[sel])) for _, sel in classes))
print('{:<16s}'.format('frac tau>0') + ''.join('{:>18.2f}'.format(np.nanmean(tau_cols[sel] > 0)) for _, sel in classes))


## Regime diagram for BBL instabilities (Wenegrat, Callies & Thomas 2018, Fig. 19)

For each water column, the background state in the BBL (mean over `0 < hab <= h_BBL`, or over a fixed height range, see `RS_LAYER`) is described by

* the **Richardson number** $Ri = N^2/\Lambda^2 = N^2 f^2/M^4$ ($\Lambda = M^2/f$ thermal-wind shear, $M^2=|\nabla_h b|$),
* the **slope parameter** $\alpha = \tan\theta\, N^2/M^2$ (bottom slope over minus the isopycnal slope),
* the **slope Burger number** $S = (N\tan\theta/f)^2 = \alpha^2/Ri$.

Regimes (paper eqs. 32-33): $q'=1-(\alpha+1)/Ri>0$ : baroclinic instability (**BI**, right of the red line $S=(Ri-1)^2/Ri$). For $q'<0$: symmetric (**SI**) if $LSP/VSP=\alpha[1-\frac{\delta^2}{Ri}(1-\alpha/Ri)]<1$ (blue line), otherwise centrifugal (**CI**) if $\alpha/Ri>1$ (left of the green line $S=Ri$, zero absolute vertical vorticity) or mixed (**SI/CI**). In the background of the diagram $\delta^2/Ri=(f/N)^2=0.01$ as in the paper; for the columns the actual $(f/N)^2$ is used. $N^2$ is the model's `bvf` (adiabatic gradient), $M^2=|\nabla_h b|$ of the mean buoyancy. Only columns with $N^2>0$ and $M^2>0$ in the layer are plotted; the fraction of columns with a statically unstable layer mean ($N^2\le0$, i.e. $Ri<0$, outside the regime diagram) is given in the last column of the table.


In [ ]:
# --- Regime diagram (Wenegrat, Callies & Thomas 2018, Fig. 19) with the columns of the simulation ------------------------
from matplotlib.colors import ListedColormap, LogNorm

RS_LAYER = None            # None: BBL mean (0 < hab <= h_BBL); or (hab_min, hab_max) in m, e.g. (0., 50.)
RS_SUBSAMPLE = 20000       # number of columns drawn as points in the linear (paper-like) panel

lay = in_bbl if RS_LAYER is None else np.broadcast_to(((hab_grid >= RS_LAYER[0]) & (hab_grid <= RS_LAYER[1]))[None, :], in_bbl.shape)


def layer_mean(f):
    with warnings.catch_warnings():
        warnings.simplefilter('ignore', RuntimeWarning)
        return np.nanmean(np.where(lay, f, np.nan), axis=1)


N2_L = layer_mean(H['N2'])                                  # N^2 = model bvf (adiabatic gradient), averaged over the layer
M2_L = layer_mean(H['M2'])                                  # |grad_h b|
tan_th = tan_th_cols                                           # bottom slope
f_c = np.abs(simul.f.reshape(-1)[cols])
valid = np.isfinite(N2_L) & np.isfinite(M2_L) & (N2_L > 0) & (M2_L > 0) & np.isfinite(tan_th)
with np.errstate(divide='ignore', invalid='ignore'):
    Ri_c = N2_L * f_c**2 / M2_L**2                             # Richardson number N2 f2 / M4
    al_c = tan_th * N2_L / M2_L                                # slope parameter alpha
    S_c = N2_L * tan_th**2 / f_c**2                            # slope Burger number (N tan(theta) / f)^2  (= alpha^2 / Ri)
    d2_c = f_c**2 / N2_L                                       # delta^2 / Ri = (f/N)^2
Ri_c, al_c, S_c = [np.where(valid, a, np.nan) for a in (Ri_c, al_c, S_c)]
unst = np.isfinite(N2_L) & (N2_L <= 0)                         # statically unstable layer mean (Ri < 0): outside the regime diagram

REGIMES = ['BI', 'SI', 'CI', 'SI/CI']


def regime(Ri, alpha, d2):
    """0 BI, 1 SI, 2 CI, 3 SI/CI (paper eqs. 32-33)."""
    with np.errstate(divide='ignore', invalid='ignore'):
        q = 1. - (alpha + 1.) / Ri
        lsp = alpha * (1. - d2 * (1. - alpha / Ri))
        return np.where(q > 0, 0, np.where(lsp < 1, 1, np.where(alpha / Ri > 1, 2, 3)))


reg_c = np.where(valid, regime(Ri_c, al_c, d2_c), -1)

# Fractions of the valid columns in each regime, per region / class
groups = [(n, s) for n, s in regions.items()]
if 'classes' in globals():
    groups += [(('w\'b\'>0' if k == 0 else 'w\'b\'<0') + ' (BBL mean)', sel) for k, (_, sel) in enumerate(classes)]
print('Layer: %s;  valid columns (N2>0, M2>0): %d of %d' % ('BBL mean' if RS_LAYER is None else str(RS_LAYER), valid.sum(), ncol))
print('{:<26s}{:>9s}'.format('', 'n valid') + ''.join('{:>8s}'.format(r) for r in REGIMES) + '{:>12s}{:>10s}{:>10s}'.format('median Ri', 'med alpha', 'med S') + '{:>12s}'.format('N2<=0 (Ri<0)'))
for name, sel in groups:
    v = sel & valid
    if v.sum() == 0:
        print('{:<26s}{:>9d}'.format(name[:25], 0) + ' ' * 62 + '{:>12.3f}'.format(np.mean(unst[sel]))); continue
    print('{:<26s}{:>9d}'.format(name[:25], v.sum()) + ''.join('{:>8.2f}'.format(np.mean(reg_c[v] == k)) for k in range(4))
          + '{:>12.3g}{:>10.3g}{:>10.3g}'.format(np.median(Ri_c[v]), np.median(al_c[v]), np.median(S_c[v]))
          + '{:>12.3f}'.format(np.mean(unst[sel])))


def draw_regimes(ax, Ri, S):
    """Background of the regime diagram on the (Ri, S) grid vectors; lines as in Fig. 19."""
    RR, SS = np.meshgrid(Ri, S)
    AL = np.sqrt(SS * RR)
    reg = regime(RR, AL, 0.01)
    ax.pcolormesh(Ri, S, reg, cmap=ListedColormap(['#ececec', '#8c8c8c', '#bdbdbd', '#d6d6d6']), vmin=-0.5, vmax=3.5, shading='auto')
    lsp = np.where(1. - (AL + 1.) / RR <= 0, AL * (1. - 0.01 * (1. - AL / RR)) - 1., np.nan)
    Rr = Ri[Ri >= 1.]
    ax.plot(Rr, (Rr - 1.)**2 / Rr, 'r--', lw=2)                                  # q' = 0
    ax.plot(Ri, Ri, 'g--', lw=2)                                                 # 1 - alpha/Ri = 0
    ax.contour(Ri, S, lsp, [0.], colors='b', linestyles='--', linewidths=2)       # LSP/VSP = 1 (theory)
    for a_ in (1., 2.):                                                           # alpha = 1, 2 (inside BI)
        R_ = Ri[Ri >= 1. + a_]
        ax.plot(R_, a_**2 / R_, 'k:', lw=1.5)


set_markers = []
cols_m = ['k', 'tab:orange', 'tab:purple', 'tab:green', 'tab:brown', 'tab:red', 'tab:blue', 'tab:cyan']
for k, (name, sel) in enumerate(groups):
    v = sel & valid
    if v.sum() > 10:
        set_markers.append((name, v, cols_m[k % len(cols_m)], 'os^Dvp<>'[k % 8]))

with plt.rc_context({'font.size': 14, 'axes.labelsize': 15}):
    fig, axs = plt.subplots(1, 2, figsize=(21, 8.8), constrained_layout=True)

    # a) paper-like linear diagram (Ri 0-5, S 0-2): columns as grey-black points
    ax = axs[0]
    Ri_g, S_g = np.linspace(0.005, 5., 700), np.linspace(0., 2., 500)
    draw_regimes(ax, Ri_g, S_g)
    idx = np.flatnonzero(valid)
    idx = np.random.default_rng(0).choice(idx, min(RS_SUBSAMPLE, idx.size), replace=False)
    ax.plot(Ri_c[idx], S_c[idx], '.', ms=2.5, color='k', alpha=0.35, rasterized=True)
    inr = valid & (Ri_c <= 5.) & (S_c <= 2.)
    ax.set_xlim(0, 5); ax.set_ylim(0, 2)
    ax.set_xlabel('Richardson number, Ri'); ax.set_ylabel('Slope Burger number, S')
    ax.set_title('a) as Fig. 19 of Wenegrat et al. (2018); %.1f %% of the valid columns in this range' % (100. * inr.sum() / max(valid.sum(), 1)))
    for txt, x, y in [('CI', 0.7, 1.7), ('SI/CI', 1.9, 1.4), ('BI', 3.4, 0.45), ('SI', 0.45, 0.3)]:
        ax.text(x, y, txt, fontsize=16)

    # b) the same diagram on log axes, with the 2D histogram of the columns and the medians of the groups
    ax = axs[1]
    Ri_g, S_g = np.logspace(-1, 4.5, 900), np.logspace(-4.5, 1., 700)
    draw_regimes(ax, Ri_g, S_g)
    ax.set_xscale('log'); ax.set_yscale('log')
    hh = ax.hist2d(Ri_c[valid], S_c[valid], bins=[np.logspace(-1, 4.5, 90), np.logspace(-4.5, 1., 80)], cmin=1,
                   norm=LogNorm(), cmap='viridis', alpha=0.9)
    fig.colorbar(hh[3], ax=ax, label='number of columns', shrink=0.85)
    for name, v, col, mk in set_markers:
        q = [np.percentile(Ri_c[v], [25, 50, 75]), np.percentile(S_c[v], [25, 50, 75])]
        ax.errorbar(q[0][1], q[1][1], xerr=[[q[0][1] - q[0][0]], [q[0][2] - q[0][1]]], yerr=[[q[1][1] - q[1][0]], [q[1][2] - q[1][1]]],
                    fmt=mk, color=col, mec='w', ms=11, capsize=3, lw=1.8, label=name)
    ax.set_xlim(0.1, 10**4.5); ax.set_ylim(10**-4.5, 10.)
    ax.set_xlabel('Richardson number, Ri'); ax.set_ylabel('Slope Burger number, S')
    ax.set_title('b) log axes: histogram of the columns, medians and 25-75 %% ranges (%s)' % ('BBL mean' if RS_LAYER is None else 'hab %g-%g m' % tuple(RS_LAYER)))
    ax.legend(fontsize=11, loc='lower left')
    for txt, x, y in [('CI', 0.15, 3.), ('SI/CI', 0.6, 3.), ('BI', 3e3, 3e-1), ('SI', 0.15, 1e-3)]:
        ax.text(x, y, txt, fontsize=16)
    fig.suptitle(simul.simul + r'   (red: $q^\prime=0$, green: $1-\alpha/Ri=0$, blue dashed: LSP/VSP=1 for $\delta^2/Ri=0.01$, dotted: $\alpha=1,2$)', fontsize=14)
    plt.savefig(fifig + 'regime_diagram_Ri_S_' + simul.simul + '.png', dpi=200, bbox_inches='tight')
    plt.show()


In [ ]:
# --- Maps of the amplitude of the buoyancy advection terms at the lowest model level -----------------------------
# |b_adv|, |b_adv^mean|, |b_adv^eddy| (and |b_rhs| for reference) on the first sigma level above the sea floor.
from matplotlib.colors import LogNorm

kbot = 0                                                    # index of the lowest rho-level (0 = bottom)
MAP_TERMS = [('b_adv', r'$|b_{adv}|$ (total advection)'),
             ('b_adv_mean', r'$|b^{mean}_{adv}|$'),
             ('b_eddy', r'$|b^{eddy}_{adv}|$ (total - mean)'),
             ('b_rhs', r'$|b_{rhs}|$ (mixing)')]
MAP_PCT = 99.5                                              # colour range up to this percentile of the |amplitude|
MAP_VMIN_FACTOR = 1e-3                                      # colour range spans this factor below the maximum

# `bot` (lowest-level values of b_adv, b_adv_mean, b_rhs) was filled when the terms were remapped
bot['b_eddy'] = bot['b_adv'] - bot['b_adv_mean']              # NaN where |b_adv| > BMASK
ocean = (simul.mask > 0) & (simul.topo >= TOPO_MIN if TOPO_MIN is not None else True)
if TOPO_MAX is not None:
    ocean &= simul.topo < TOPO_MAX

xkm = np.cumsum(1. / simul.pm[:, ny // 2]) / 1e3            # distance along xi / eta [km]
ykm = np.cumsum(1. / simul.pn[nx // 2, :]) / 1e3

amp = {k: np.where(ocean, np.abs(bot[k]), np.nan) for k, _ in MAP_TERMS}
vmax = np.nanpercentile(np.concatenate([amp[k][ocean] for k in ('b_adv', 'b_adv_mean', 'b_eddy')]), MAP_PCT)
print('lowest-level amplitude of the terms [m s-3] (domain used):')
for k, label in MAP_TERMS:
    i, j = np.unravel_index(np.nanargmax(amp[k]), amp[k].shape)
    print('  %-11s max %.2e at (i=%d, j=%d), topo %.0f m | %.1f-th percentile %.2e | median %.2e'
          % (k, amp[k][i, j], i, j, simul.topo[i, j], MAP_PCT, np.nanpercentile(amp[k][ocean], MAP_PCT),
             np.nanmedian(amp[k][ocean])))

fig, axs = plt.subplots(2, 2, figsize=(17, 11), sharex=True, sharey=True, constrained_layout=True)
for ax, (k, label) in zip(axs.ravel(), MAP_TERMS):
    pc = ax.pcolormesh(xkm, ykm, amp[k].T, shading='auto',
                       norm=LogNorm(vmin=vmax * MAP_VMIN_FACTOR, vmax=vmax), cmap='magma_r')
    ax.contour(xkm, ykm, np.where(simul.mask > 0, simul.topo, np.nan).T,
               levels=np.arange(250., 6000., 250.), colors='k', linewidths=0.3)
    ax.set_title(label)
    fig.colorbar(pc, ax=ax, extend='both', label=r'm s$^{-3}$')
for ax in axs[1]:
    ax.set_xlabel(r'km in $\xi$-direction')
for ax in axs[:, 0]:
    ax.set_ylabel(r'km in $\eta$-direction')
fig.suptitle('Amplitude of the buoyancy-budget terms at the lowest model level (time mean)')
os.makedirs(fifig, exist_ok=True)
plt.savefig(fifig + 'map_bottom_level_advection_' + simul.simul.split('/')[-1][:40] + '.png', dpi=150)
plt.show()
